# Second task family: random-transition m-th-order Markov sources, V = 8, entropy matched

Thirty 12-layer models (the k-gram architecture, optimiser and schedule) on random back-off Markov sources of order m ∈ {1, 2, 3, 4, 5, 6}, five seeds each, every source tempered to a conditional entropy of 1.00 nats and a marginal entropy within 0.1 nats of ln 8, so only the order varies. Each run trains 10,000 steps on fresh data, then measures the canonical σ₁ profile, branch rotation and identity skip by region and per block, early exit, the exact held-out floor under the true source and the learned gap.

**Fix (20 Sept, evening).** The first sweep ran 2,000 steps because the learnability cell had changed the shared training configuration in the kernel; the run cell now resets it and asserts 10,000 steps, and outputs go to `markov_v1_10k/`. The learnability cell can be skipped this time.

**Time.** About 13 min per run on an A100 (about 6.5 h for all 30), about 6 min per run on the RTX PRO 6000 (3 h). Runs are ordered seed-major, so after the first six runs (about 80 min) there is a complete order ladder at one seed. A T4 is too slow.

**Outputs.** `MyDrive/hourglass/markov_v1/markov/markov_{m}_s{seed}.json` (one per run, written the moment the run finishes), fp16 checkpoints in `markov/ckpt/`, source caches in `data/`, and `markov_v1_results.zip` on Drive and in `/content` at the end. A disconnect loses nothing; re-run the run cell and finished runs are skipped.

**Optional first.** The learnability cell trains the full model for 2,000 steps at orders 4, 5 and 6 (one seed, about 8 min on an A100) and prints the held-out gap to the floor and the margin below the best order-(m−1) predictor; if order 6 has not moved below its order-5 floor by then, set `MARKOV_ORDERS = "1,2,3,4,5"` in the paths cell before the sweep.

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content')
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
import torch; print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "no GPU")

In [ ]:
H = "/content/drive/MyDrive/hourglass"
os.environ["MARKOV_OUT"] = f"{H}/markov_v1_10k"      # new folder: the 20 Sept run in markov_v1/ was 2,000 steps (quick-cell bug) and is kept
os.environ["REPRO_OUT"] = f"{H}/reprofile_v2"     # only so reprofile_v2 imports with its usual root; nothing is read from it
os.makedirs(f"{H}/markov_v1_10k/markov", exist_ok=True); os.makedirs(f"{H}/markov_v1_10k/data", exist_ok=True)
MARKOV_ORDERS = "1,2,3,4,5,6"                      # drop 6 here if the learnability cell says so
print("outputs ->", os.environ["MARKOV_OUT"])

In [ ]:
%%writefile /content/survey_sigma1_v2.py
"""
survey_sigma1_v2.py  --  one profiler, one protocol, every model
================================================================

Measures the quantity the paper defines, for every model in the survey and census
panels, with one script:

    sigma_1 of the token-local Jacobian  J_l,t = d h_{l+1,t} / d h_{l,t}

i.e. perturb the hidden state at ONE position t at the input of block l, with every
other position held fixed at its natural-forward value, and read the change at the
same position t at the output of block l. sigma_1 is the leading singular value of
that d x d map, found by power iteration on J^T J with exact automatic
differentiation (torch.func.jvp for J v, reverse-mode autograd for J^T u), 50
iterations, four random restarts per position, max over restarts.

Inputs are natural: WikiText-103 validation text for language models (5 sequences of
128 tokens, 8 probe positions each), CIFAR-100 test images for ConvNeXt, ViT, DINOv2
and MLP-Mixer, LibriSpeech test-clean for Whisper and wav2vec2. Float32, TF32 off.

The profile is over blocks (block 0 = first transformer block / first ConvNeXt block).
R_ex0 = mean sigma_1 over the middle third of blocks 1..L-1 divided by the mean over
the two edge thirds; late uptick = mean(last third) / mean(middle third); a model is
hourglass-positive if R_ex0 < 0.80 and late uptick > 1.0 (the paper's rule). The May
census vertex/median R_ex0, spectral contrast C and quadratic curvature a are written
too, so the scrambling census 2x2 can use this geometry.

Why this exists: the April survey values came from five or more profiling scripts
whose sigma_1 values are not comparable; the 29 April reprofile pushed each token
through its layer alone (sequence length 1) on random ids and iterated on J^T rather
than J^T J; the May census used finite-difference sigma_1 of the whole-sequence layer
Jacobian on random ids. Three different quantities, three different classifications.
This script replaces all of them and, with --protocol-panel, measures the alternative
protocols on eight models so the dependence can be shown (Extended Data figure).

Modes
  --dtype float16        same inputs and settings, model in float16 (Fig. 1d); uses the
                         finite-difference protocol of the April note (relative step --fd-eps,
                         default 1e-3) because forward-mode AD is unreliable in half precision,
                         and records non-finite outputs instead of stopping
  --branch               sigma_1 of J - I, the branch Jacobian (Extended Data Fig. 5);
                         decoders, Mamba and ConvNeXt only (pre-norm residual blocks)
  --context incontext    canonical (default)
  --context isolated     token pushed through the block alone, kwargs sliced to t
  --context fullseq      sigma_1 of the whole-sequence layer Jacobian
  --inputs natural       canonical (default)      --inputs random   random ids/pixels
  --protocol-panel       the 8-model x 4-protocol sub-run
  --estimator rho        power iteration on J instead of J^T J: the spectral radius, which is
                         what the earlier finite-difference variant measured (2-9x smaller than
                         sigma_1 on these layers, and it flattens profiles); tag suffix -rho
  --validate             exact Jacobians at 5 blocks x 3 positions x 2 inputs: exact sigma_1 and rho
                         against both estimators and 64 random probes (Fig. 1b,c); sequence models
  --panel april|union    the 33 April survey models, or April + May census extras (45)

Usage (shell)
  python survey_sigma1_v2.py --panel union --results-dir results/survey_sigma1_v2
  python survey_sigma1_v2.py --models gpt2 --smoke            # reference check, prints Fig. 1a anchors
  python survey_sigma1_v2.py --protocol-panel
  python survey_sigma1_v2.py --dtype float16 --models EleutherAI/pythia-2.8b EleutherAI/pythia-6.9b EleutherAI/pythia-12b
  python survey_sigma1_v2.py --branch --models gpt2 microsoft/phi-2 EleutherAI/pythia-410m ...

Usage (Colab cell): set env vars SURVEY_PANEL, SURVEY_MODELS (space-separated),
SURVEY_RESULTS_DIR, SURVEY_DTYPE, SURVEY_BRANCH=1, SURVEY_CONTEXT, SURVEY_INPUTS,
SURVEY_PROTOCOL_PANEL=1, SURVEY_SMOKE=1, SURVEY_QUICK=1, SURVEY_NO_SKIP=1, SURVEY_ZIP=1,
SURVEY_FD_EPS, SURVEY_PURGE_CACHE=1 (delete each model's Hub cache after use; Colab's disk fills
at about 100 GB of weights),
then exec(open("survey_sigma1_v2.py").read()). The notebook's -f argument is ignored.

Resume-safe: an existing JSON for (model, protocol) is skipped unless --no-skip.
Output: <results-dir>/<protocol>/sigma1_<model>.json, plus survey_summary.json/.csv/.md
per protocol directory, plus survey_results.zip with --zip.
"""

import os, sys, json, time, gc, math, argparse, shutil, datetime, traceback
import numpy as np
import torch

# ---------------------------------------------------------------- precision
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
SEQ_LEN = 128
N_INPUTS = 5
N_POSITIONS = 8
N_RESTARTS = 4
PI_ITERS = 50
PI_TOL = 1e-6
N_BOOT = 500
AUDIO_SECONDS = 10.0
VERSION = "survey-v2.2-2026-09-15"


def log(msg):
    print(msg, flush=True)


def _resolve_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:
            tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
    return tok


# ---------------------------------------------------------------- registry
# (hf_id, arch, label, family, params_M, april_panel)
REGISTRY = [
    ("EleutherAI/pythia-70m",   "pythia", "Pythia-70M",   "AR decoder", 70,    True),
    ("EleutherAI/pythia-160m",  "pythia", "Pythia-160M",  "AR decoder", 160,   True),
    ("EleutherAI/pythia-410m",  "pythia", "Pythia-410M",  "AR decoder", 410,   True),
    ("EleutherAI/pythia-1b",    "pythia", "Pythia-1B",    "AR decoder", 1000,  True),
    ("EleutherAI/pythia-1.4b",  "pythia", "Pythia-1.4B",  "AR decoder", 1400,  True),
    ("EleutherAI/pythia-2.8b",  "pythia", "Pythia-2.8B",  "AR decoder", 2800,  True),
    ("EleutherAI/pythia-6.9b",  "pythia", "Pythia-6.9B",  "AR decoder", 6900,  True),
    ("EleutherAI/pythia-12b",   "pythia", "Pythia-12B",   "AR decoder", 12000, True),
    ("gpt2",                    "gpt2",   "GPT-2 124M",   "AR decoder", 124,   True),
    ("gpt2-large",              "gpt2",   "GPT-2 Large",  "AR decoder", 774,   True),
    ("gpt2-xl",                 "gpt2",   "GPT-2 XL",     "AR decoder", 1558,  True),
    ("microsoft/phi-2",         "phi",    "Phi-2",        "AR decoder", 2700,  True),
    ("mistralai/Mistral-7B-v0.1", "mistral", "Mistral-7B", "AR decoder", 7200, True),
    ("Qwen/Qwen2.5-0.5B",       "qwen",   "Qwen2.5-0.5B", "AR decoder", 494,   True),
    ("Qwen/Qwen2.5-1.5B",       "qwen",   "Qwen2.5-1.5B", "AR decoder", 1540,  True),
    ("Qwen/Qwen2.5-3B",         "qwen",   "Qwen2.5-3B",   "AR decoder", 3090,  True),
    ("Qwen/Qwen2.5-7B",         "qwen",   "Qwen2.5-7B",   "AR decoder", 7610,  True),
    ("Qwen/Qwen2.5-14B",        "qwen",   "Qwen2.5-14B",  "AR decoder", 14700, True),
    ("meta-llama/Llama-3.2-1B", "llama",  "Llama-3.2-1B", "AR decoder", 1240,  True),
    ("meta-llama/Llama-3.2-3B", "llama",  "Llama-3.2-3B", "AR decoder", 3210,  True),
    ("google/gemma-2-2b",       "gemma2", "Gemma-2-2B",   "AR decoder", 2610,  True),
    ("facebook/convnext-tiny-224",  "convnext", "ConvNeXt-tiny",  "ConvNet (vision)", 28,  True),
    ("facebook/convnext-small-224", "convnext", "ConvNeXt-small", "ConvNet (vision)", 50,  True),
    ("facebook/convnext-base-224",  "convnext", "ConvNeXt-base",  "ConvNet (vision)", 89,  True),
    ("facebook/convnext-large-224", "convnext", "ConvNeXt-large", "ConvNet (vision)", 198, True),
    ("state-spaces/mamba-130m-hf",  "mamba", "Mamba-130M", "SSM", 130,  True),
    ("state-spaces/mamba-1.4b-hf",  "mamba", "Mamba-1.4B", "SSM", 1400, True),
    ("state-spaces/mamba-2.8b-hf",  "mamba", "Mamba-2.8B", "SSM", 2800, True),
    ("google-bert/bert-base-uncased", "bert",    "BERT-base",    "Masked encoder", 110, True),
    ("FacebookAI/roberta-base",      "roberta", "RoBERTa-base", "Masked encoder", 125, True),
    ("google/vit-base-patch16-224",  "vit",     "ViT-base",     "Vision supervised", 86, True),
    ("facebook/dinov2-base",         "dinov2",  "DINOv2-base",  "Vision SSL", 86, True),
    ("openai/whisper-small",         "whisper", "Whisper-small", "Audio encoder", 88, True),
    # May census extras
    ("google-t5/t5-small",   "t5", "T5-small (encoder)",   "Encoder-decoder", 60,  False),
    ("google-t5/t5-base",    "t5", "T5-base (encoder)",    "Encoder-decoder", 220, False),
    ("google-t5/t5-large",   "t5", "T5-large (encoder)",   "Encoder-decoder", 770, False),
    ("google/flan-t5-large", "t5", "Flan-T5-large (encoder)", "Encoder-decoder", 780, False),
    ("facebook/wav2vec2-base-960h", "wav2vec2", "wav2vec2-base", "Audio encoder", 95, False),
    ("google-bert/bert-large-uncased", "bert",    "BERT-large",    "Masked encoder", 340, False),
    ("FacebookAI/roberta-large",      "roberta", "RoBERTa-large", "Masked encoder", 355, False),
    ("state-spaces/mamba-370m-hf",    "mamba",   "Mamba-370M",    "SSM", 370, False),
    ("Qwen/Qwen2-0.5B",               "qwen",    "Qwen2-0.5B",    "AR decoder", 494, False),
    ("mistralai/Mistral-7B-Instruct-v0.2", "mistral", "Mistral-7B-Instruct", "AR decoder", 7200, False),
    ("timm/mixer_b16_224.goog_in21k_ft_in1k", "mlp_mixer", "MLP-Mixer-B", "MLP-Mixer (vision)", 60,  False),
    ("timm/mixer_l16_224.goog_in21k_ft_in1k", "mlp_mixer", "MLP-Mixer-L", "MLP-Mixer (vision)", 208, False),
]
CFG = {r[0]: dict(hf_id=r[0], arch=r[1], label=r[2], family=r[3], params_M=r[4], april=r[5]) for r in REGISTRY}

PROTOCOL_PANEL_MODELS = ["EleutherAI/pythia-160m", "EleutherAI/pythia-6.9b", "gpt2", "Qwen/Qwen2.5-7B",
                         "meta-llama/Llama-3.2-1B", "state-spaces/mamba-130m-hf",
                         "facebook/convnext-tiny-224", "facebook/convnext-base-224"]
PROTOCOL_PANEL = [("incontext", "natural"), ("incontext", "random"), ("isolated", "random"), ("fullseq", "random")]

DECODERS = ("gpt2", "pythia", "llama", "qwen", "mistral", "gemma2", "phi")
BRANCH_OK = DECODERS + ("mamba", "convnext")
TEXT_ARCHES = DECODERS + ("mamba", "bert", "roberta", "t5")
IMAGE_ARCHES = ("convnext", "vit", "dinov2", "mlp_mixer")
AUDIO_ARCHES = ("whisper", "wav2vec2")

# v35 Fig. 1a, exact full-Jacobian sigma_1 on GPT-2 124M (text inputs), for the smoke check
GPT2_REFERENCE = {0: 9.8, 6: 1.1, 11: 2.5}


# ---------------------------------------------------------------- arguments
def parse_args(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--models", nargs="*", default=None)
    ap.add_argument("--panel", default=None, choices=["april", "union"])
    ap.add_argument("--results-dir", default=None)
    ap.add_argument("--dtype", default=None, choices=["float32", "float16", "bfloat16"])
    ap.add_argument("--branch", action="store_true")
    ap.add_argument("--context", default=None, choices=["incontext", "isolated", "fullseq"])
    ap.add_argument("--inputs", default=None, choices=["natural", "random"])
    ap.add_argument("--protocol-panel", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--no-skip", action="store_true")
    ap.add_argument("--zip", action="store_true")
    ap.add_argument("--n-inputs", type=int, default=None)
    ap.add_argument("--n-positions", type=int, default=None)
    ap.add_argument("--restarts", type=int, default=None)
    ap.add_argument("--iters", type=int, default=None)
    ap.add_argument("--fd-eps", type=float, default=None, help="relative finite-difference step for reduced-precision runs")
    ap.add_argument("--purge-cache", action="store_true", help="delete each model's Hub cache after profiling it")
    ap.add_argument("--force-fd", action="store_true", help="use the finite-difference protocol in float32 too (matched control for --dtype float16)")
    ap.add_argument("--estimator", default=None, choices=["sigma1", "rho"],
                    help="sigma1 (power iteration on J^T J, default) or rho (power iteration on J: the spectral radius the earlier finite-difference variant estimated)")
    ap.add_argument("--validate", action="store_true",
                    help="materialise the exact token-local Jacobian at five blocks x three positions x two inputs and compare exact sigma1/rho with the estimators and a 64-direction random-probe baseline (Fig. 1b,c)")
    args, unknown = ap.parse_known_args(argv)
    env = os.environ
    if args.models is None and env.get("SURVEY_MODELS"):
        args.models = env["SURVEY_MODELS"].split()
    args.panel = args.panel or env.get("SURVEY_PANEL") or "union"
    args.results_dir = args.results_dir or env.get("SURVEY_RESULTS_DIR") or "results/survey_sigma1_v2"
    args.dtype = args.dtype or env.get("SURVEY_DTYPE") or "float32"
    args.branch = args.branch or env.get("SURVEY_BRANCH") == "1"
    args.context = args.context or env.get("SURVEY_CONTEXT") or "incontext"
    args.inputs = args.inputs or env.get("SURVEY_INPUTS") or "natural"
    args.protocol_panel = args.protocol_panel or env.get("SURVEY_PROTOCOL_PANEL") == "1"
    args.smoke = args.smoke or env.get("SURVEY_SMOKE") == "1"
    args.quick = args.quick or env.get("SURVEY_QUICK") == "1"
    args.no_skip = args.no_skip or env.get("SURVEY_NO_SKIP") == "1"
    args.zip = args.zip or env.get("SURVEY_ZIP") == "1"
    args.n_inputs = args.n_inputs or int(env.get("SURVEY_N_INPUTS", N_INPUTS))
    args.n_positions = args.n_positions or int(env.get("SURVEY_N_POSITIONS", N_POSITIONS))
    args.restarts = args.restarts or int(env.get("SURVEY_RESTARTS", N_RESTARTS))
    args.iters = args.iters or int(env.get("SURVEY_ITERS", PI_ITERS))
    args.fd_eps = args.fd_eps or float(env.get("SURVEY_FD_EPS", 1e-3))
    args.purge_cache = args.purge_cache or env.get("SURVEY_PURGE_CACHE") == "1"
    args.force_fd = args.force_fd or env.get("SURVEY_FORCE_FD") == "1"
    args.estimator = args.estimator or env.get("SURVEY_ESTIMATOR") or "sigma1"
    args.validate = args.validate or env.get("SURVEY_VALIDATE") == "1"
    if args.quick:
        args.n_inputs, args.n_positions, args.restarts, args.iters = 1, 2, 1, 6
    return args


def protocol_tag(context, inputs, branch, dtype, fd=False, estimator="sigma1"):
    return (f"{context}-{inputs}-{'branch' if branch else 'block'}-{dtype}" + ("-fd" if fd else "")
            + ("-rho" if estimator == "rho" else ""))


def safe_name(hf_id):
    return hf_id.replace("/", "_").replace(".", "_")


# ---------------------------------------------------------------- data
def _wikitext_validation_texts():
    token = os.environ.get("HF_TOKEN")
    try:
        from huggingface_hub import hf_hub_download
        import pandas as pd
        p = hf_hub_download("Salesforce/wikitext", "wikitext-103-raw-v1/validation-00000-of-00001.parquet",
                            repo_type="dataset", token=token)
        return pd.read_parquet(p)["text"].tolist()
    except Exception as e:
        log(f"  parquet download failed ({e}); falling back to datasets")
        from datasets import load_dataset
        return list(load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")["text"])


_TEXT_CACHE = {}


def natural_text_ids(tokenizer, n_seqs, seq_len=SEQ_LEN):
    """n_seqs sequences of seq_len tokens, spread evenly through WikiText-103 validation."""
    if "text" not in _TEXT_CACHE:
        texts = _wikitext_validation_texts()
        _TEXT_CACHE["text"] = "\n".join(t for t in texts if len(t.strip()) > 50)
    text = _TEXT_CACHE["text"]
    # tokenise a slice around each of n_seqs evenly spaced character offsets (cheap)
    n_chars = len(text)
    offsets = np.linspace(0.05, 0.95, n_seqs) * n_chars
    seqs = []
    for off in offsets:
        chunk = text[int(off): int(off) + 20 * seq_len]
        ids = tokenizer.encode(chunk, add_special_tokens=False)
        if len(ids) < seq_len:
            ids = tokenizer.encode(text[int(off): int(off) + 200 * seq_len], add_special_tokens=False)
        seqs.append(ids[:seq_len])
    return torch.tensor(seqs, dtype=torch.long)


def random_text_ids(vocab_size, n_seqs, seq_len=SEQ_LEN):
    g = torch.Generator().manual_seed(SEED)
    return torch.randint(0, vocab_size, (n_seqs, seq_len), generator=g)


def _cifar100_test_images(n):
    try:
        import torchvision
        ds = torchvision.datasets.CIFAR100(root="./data_cifar100", train=False, download=True)
        idx = np.linspace(0, len(ds) - 1, n).astype(int)
        return [ds[int(i)][0].convert("RGB") for i in idx]
    except Exception as e:
        log(f"  torchvision CIFAR-100 failed ({e}); falling back to datasets")
        from datasets import load_dataset
        ds = load_dataset("uoft-cs/cifar100", split="test")
        idx = np.linspace(0, len(ds) - 1, n).astype(int)
        return [ds[int(i)]["img"].convert("RGB") for i in idx]


def _librispeech_waveforms(n, seconds=AUDIO_SECONDS):
    """n LibriSpeech test-clean utterances >= 4 s, truncated to `seconds`, 16 kHz float tensors."""
    out = []
    try:
        import torchaudio
        ds = torchaudio.datasets.LIBRISPEECH("./data_librispeech", url="test-clean", download=True)
        idx = np.linspace(0, len(ds) - 1, 4 * n).astype(int)
        for i in idx:
            wav, sr = ds[int(i)][0], ds[int(i)][1]
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            wav = wav[0]
            if wav.numel() >= 4 * 16000:
                out.append(wav[: int(seconds * 16000)].float())
            if len(out) == n:
                break
    except Exception as e:
        log(f"  torchaudio LibriSpeech failed ({e}); falling back to datasets streaming")
        from datasets import load_dataset
        ds = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)
        for ex in ds:
            wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32)
            if ex["audio"]["sampling_rate"] != 16000:
                continue
            if wav.numel() >= 4 * 16000:
                out.append(wav[: int(seconds * 16000)])
            if len(out) == n:
                break
    return out


# ---------------------------------------------------------------- loading
def _from_pretrained(cls, hf_id, dtype, **kw):
    """dtype= on transformers >= 4.56 else torch_dtype=; drops attn_implementation if refused."""
    for dkey in ("dtype", "torch_dtype"):
        try:
            return cls.from_pretrained(hf_id, **{dkey: dtype}, **kw)
        except TypeError as e:
            if "attn_implementation" in str(e) and "attn_implementation" in kw:
                kw = {k: v for k, v in kw.items() if k != "attn_implementation"}
                return cls.from_pretrained(hf_id, **{dkey: dtype}, **kw)
            if dkey == "torch_dtype":
                raise
        except ValueError as e:
            if "attn_implementation" in kw and ("eager" in str(e) or "attn_implementation" in str(e)):
                kw = {k: v for k, v in kw.items() if k != "attn_implementation"}
                return cls.from_pretrained(hf_id, **{dkey: dtype}, **kw)
            raise


LAYER_PATHS = {
    "gpt2": ["transformer.h", "h"],
    "llama": ["model.layers", "layers"], "gemma2": ["model.layers", "layers"], "phi": ["model.layers", "layers"],
    "qwen": ["model.layers", "layers"], "mistral": ["model.layers", "layers"],
    "pythia": ["gpt_neox.layers", "layers"],
    "mamba": ["backbone.layers", "layers"],
    "bert": ["encoder.layer", "encoder.layers", "layers", "layer"], "roberta": ["encoder.layer", "encoder.layers", "layers", "layer"],
    "vit": ["encoder.layer", "encoder.layers", "layers", "layer"], "dinov2": ["encoder.layer", "encoder.layers", "layers", "layer"],
    "t5": ["encoder.block", "block", "encoder.layers"],
    "whisper": ["layers", "encoder.layers"],
    "wav2vec2": ["encoder.layers", "layers"],
    "mlp_mixer": ["blocks"],
}


def _get_path(obj, path):
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj


def get_layers(model, arch):
    """The block list of a model, tolerant of the attribute renames across transformers versions."""
    if arch == "convnext":
        enc = model.convnext.encoder if hasattr(model, "convnext") else model.encoder
        return [blk for stage in enc.stages for blk in stage.layers]
    for path in LAYER_PATHS[arch]:
        try:
            layers = _get_path(model, path)
            if isinstance(layers, (torch.nn.ModuleList, torch.nn.Sequential)) and len(layers) > 0:
                return list(layers)
        except AttributeError:
            continue
    # generic fallback: the longest ModuleList of block-like modules
    best = None
    for name, mod in model.named_modules():
        if isinstance(mod, (torch.nn.ModuleList, torch.nn.Sequential)) and len(mod) > 1 and \
                any(k in type(mod[0]).__name__ for k in ("Layer", "Block", "Decoder", "Encoder")):
            if best is None or len(mod) > len(best[1]):
                best = (name, mod)
    if best is None:
        raise ValueError(f"cannot find the block list for {arch}")
    log(f"  (block list found by search at '{best[0]}')")
    return list(best[1])


def load_bundle(cfg, dtype_name):
    """Returns dict(model, layers, arch, kind, forward, tokenizer/processor, vocab_size, cls_offset)."""
    arch, hf_id = cfg["arch"], cfg["hf_id"]
    token = os.environ.get("HF_TOKEN")
    dtype = getattr(torch, dtype_name)
    log(f"  Loading {hf_id} ({arch}, {dtype_name})...")
    b = dict(arch=arch, hf_id=hf_id, cfg=cfg, dtype=dtype, cls_offset=0)
    if arch in DECODERS or arch == "mamba":
        from transformers import AutoModelForCausalLM, AutoTokenizer
        kw = dict(token=token)
        if arch != "mamba":
            kw["attn_implementation"] = "eager"
        model = _from_pretrained(AutoModelForCausalLM, hf_id, dtype, **kw)
        tok = AutoTokenizer.from_pretrained(hf_id, token=token)
        b.update(kind="text", tokenizer=tok, vocab_size=int(model.config.vocab_size),
                 forward=lambda m, inp: m(input_ids=inp["input_ids"], use_cache=False))
    elif arch in ("bert", "roberta"):
        from transformers import AutoModel, AutoTokenizer
        model = _from_pretrained(AutoModel, hf_id, dtype, token=token, attn_implementation="eager")
        tok = AutoTokenizer.from_pretrained(hf_id, token=token)
        b.update(kind="text", tokenizer=tok, vocab_size=int(model.config.vocab_size),
                 forward=lambda m, inp: m(input_ids=inp["input_ids"]))
    elif arch == "t5":
        from transformers import T5EncoderModel, AutoTokenizer
        model = _from_pretrained(T5EncoderModel, hf_id, dtype, token=token)
        tok = AutoTokenizer.from_pretrained(hf_id, token=token)
        b.update(kind="text", tokenizer=tok, vocab_size=int(model.config.vocab_size),
                 forward=lambda m, inp: m(input_ids=inp["input_ids"]))
    elif arch in ("vit", "dinov2"):
        from transformers import AutoModel, AutoImageProcessor
        model = _from_pretrained(AutoModel, hf_id, dtype, token=token, attn_implementation="eager")
        proc = AutoImageProcessor.from_pretrained(hf_id, token=token)
        b.update(kind="image", processor=proc, cls_offset=1,
                 forward=lambda m, inp: m(pixel_values=inp["pixel_values"]))
        if arch == "dinov2":
            b["cls_offset"] = 1 + int(getattr(model.config, "num_register_tokens", 0) or 0)
    elif arch == "convnext":
        from transformers import ConvNextForImageClassification, AutoImageProcessor
        model = _from_pretrained(ConvNextForImageClassification, hf_id, dtype, token=token)
        proc = AutoImageProcessor.from_pretrained(hf_id, token=token)
        b.update(kind="image", processor=proc, forward=lambda m, inp: m(pixel_values=inp["pixel_values"]))
    elif arch == "mlp_mixer":
        import timm
        name = hf_id.replace("timm/", "")
        model = timm.create_model(name, pretrained=True)
        if dtype != torch.float32:
            model = model.to(dtype)
        data_cfg = timm.data.resolve_data_config({}, model=model)
        transform = timm.data.create_transform(**data_cfg)
        b.update(kind="image", timm_transform=transform, timm_input_size=data_cfg.get("input_size", (3, 224, 224)),
                 forward=lambda m, inp: m(inp["pixel_values"]))
    elif arch == "whisper":
        from transformers import WhisperModel, WhisperFeatureExtractor
        full = _from_pretrained(WhisperModel, hf_id, dtype, token=token, attn_implementation="eager")
        model = full.encoder
        fe = WhisperFeatureExtractor.from_pretrained(hf_id, token=token)
        b.update(kind="audio", feature_extractor=fe, forward=lambda m, inp: m(input_features=inp["input_features"]))
    elif arch == "wav2vec2":
        from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
        model = _from_pretrained(Wav2Vec2Model, hf_id, dtype, token=token, attn_implementation="eager")
        fe = Wav2Vec2FeatureExtractor.from_pretrained(hf_id, token=token)
        b.update(kind="audio", feature_extractor=fe, forward=lambda m, inp: m(input_values=inp["input_values"]))
    else:
        raise ValueError(arch)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    b["model"] = model
    b["layers"] = get_layers(model, arch)
    log(f"  {len(b['layers'])} blocks; {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
    return b


# ---------------------------------------------------------------- inputs
def prepare_inputs(b, inputs_kind, n_inputs):
    """List of input dicts for the model forward, each with batch 1, plus 'valid_T' hints."""
    arch, kind = b["arch"], b["kind"]
    out = []
    if kind == "text":
        if inputs_kind == "natural":
            ids = natural_text_ids(b["tokenizer"], n_inputs)
        else:
            ids = random_text_ids(b["vocab_size"], n_inputs)
        for i in range(ids.shape[0]):
            out.append({"input_ids": ids[i:i + 1].to(DEVICE)})
        desc = f"{'WikiText-103 validation' if inputs_kind == 'natural' else 'uniform random ids'}, {n_inputs} x {SEQ_LEN} tokens"
    elif kind == "image":
        if inputs_kind == "natural":
            imgs = _cifar100_test_images(n_inputs)
            if arch == "mlp_mixer":
                px = torch.stack([b["timm_transform"](im) for im in imgs])
            else:
                px = b["processor"](images=imgs, return_tensors="pt")["pixel_values"]
            desc = f"CIFAR-100 test images, {n_inputs}, {tuple(px.shape[1:])}"
        else:
            g = torch.Generator().manual_seed(SEED)
            shape = (3, 224, 224)
            if arch == "mlp_mixer":
                shape = tuple(b.get("timm_input_size", shape))
            elif arch in ("vit", "dinov2", "convnext"):
                proc = b["processor"]
                cs = getattr(proc, "crop_size", None) or getattr(proc, "size", None) or {}
                h = cs.get("height", cs.get("shortest_edge", 224)) if isinstance(cs, dict) else 224
                shape = (3, int(h), int(h))
            px = torch.randn((n_inputs,) + shape, generator=g)
            desc = f"random pixels N(0,1) in normalised space, {n_inputs}, {shape}"
        for i in range(px.shape[0]):
            out.append({"pixel_values": px[i:i + 1].to(DEVICE, dtype=b["dtype"])})
    elif kind == "audio":
        fe = b["feature_extractor"]
        if inputs_kind == "natural":
            wavs = _librispeech_waveforms(n_inputs)
            desc = f"LibriSpeech test-clean, {n_inputs} utterances, first {AUDIO_SECONDS:.0f} s"
        else:
            g = torch.Generator().manual_seed(SEED)
            wavs = [0.05 * torch.randn(int(AUDIO_SECONDS * 16000), generator=g) for _ in range(n_inputs)]
            desc = f"white noise, {n_inputs} x {AUDIO_SECONDS:.0f} s"
        for w in wavs:
            if arch == "whisper":
                feats = fe(w.numpy(), sampling_rate=16000, return_tensors="pt")["input_features"]
                valid_T = int(math.ceil(w.numel() / 16000 / 30.0 * 1500))
                out.append({"input_features": feats.to(DEVICE, dtype=b["dtype"]), "valid_T": valid_T})
            else:
                vals = fe(w.numpy(), sampling_rate=16000, return_tensors="pt")["input_values"]
                out.append({"input_values": vals.to(DEVICE, dtype=b["dtype"])})
    else:
        raise ValueError(kind)
    b["input_desc"] = desc
    log(f"  Inputs: {desc}")
    return out


# ---------------------------------------------------------------- capture
def _strip_kwargs(kw):
    out = {}
    for k, v in kw.items():
        if "past" in k or k in ("use_cache", "cache_params", "layer_past"):
            continue
        out[k] = v
    return out


def capture_layer_inputs(b, inp):
    """One forward with pre-hooks; returns per-layer (hidden[1,...], extra_args, kwargs)."""
    layers = b["layers"]
    captured = [None] * len(layers)
    hooks = []

    def make_hook(i):
        def fn(mod, args, kwargs):
            if len(args) == 0:
                kwargs = dict(kwargs)
                h = kwargs.pop("hidden_states")
                rest = ()
            else:
                h, rest = args[0], tuple(args[1:])
            captured[i] = (h.detach(), rest, _strip_kwargs(kwargs))
        return fn

    for i, l in enumerate(layers):
        hooks.append(l.register_forward_pre_hook(make_hook(i), with_kwargs=True))
    try:
        with torch.no_grad():
            b["forward"](b["model"], inp)
    finally:
        for h in hooks:
            h.remove()
    missing = [i for i, c in enumerate(captured) if c is None]
    if missing:
        raise RuntimeError(f"pre-hook never fired for blocks {missing}")
    return captured


def make_layer_call(layer, rest, kw):
    def call(X):
        out = layer(X, *rest, **kw)
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out
    return call


# ---------------------------------------------------------------- token layout
class Layout:
    """Where the tokens are in a block's hidden tensor: (1,T,d) sequences or (1,C,H,W) maps."""

    def __init__(self, hidden):
        self.dim = hidden.dim()
        if self.dim == 3:
            _, self.T, self.d = hidden.shape
        elif self.dim == 4:
            _, self.d, self.H, self.W = hidden.shape
            self.T = self.H * self.W
        else:
            raise ValueError(f"unsupported hidden shape {tuple(hidden.shape)}")

    def hw(self, p):
        return divmod(int(p), self.W)

    def get(self, hidden, positions):
        if self.dim == 3:
            return hidden[0, positions, :]                      # (B, d)
        hs, ws = zip(*[self.hw(p) for p in positions])
        return hidden[0, :, list(hs), list(ws)].T               # (B, d)

    def mask(self, positions, device):
        B = len(positions)
        if self.dim == 3:
            M = torch.zeros(B, self.T, 1, device=device)
            M[torch.arange(B), torch.as_tensor(positions)] = 1.0
        else:
            M = torch.zeros(B, 1, self.H, self.W, device=device)
            for i, p in enumerate(positions):
                h, w = self.hw(p)
                M[i, 0, h, w] = 1.0
        return M

    def broadcast(self, xb):
        return xb[:, None, :] if self.dim == 3 else xb[:, :, None, None]

    def gather(self, Y, positions):
        B = len(positions)
        if self.dim == 3:
            return Y[torch.arange(B, device=Y.device), torch.as_tensor(positions, device=Y.device), :]
        hs, ws = zip(*[self.hw(p) for p in positions])
        return Y[torch.arange(B, device=Y.device), :, list(hs), list(ws)]


def choose_positions(layout, n_pos, cls_offset=0, valid_T=None):
    """n_pos probe positions: sequences, evenly spaced from T/8 to T-1 after the CLS/register
    tokens (and within the valid span for padded audio); maps, an interior grid."""
    if layout.dim == 3:
        T = layout.T if valid_T is None else min(layout.T, valid_T)
        lo = max(cls_offset, T // 8)
        hi = T - 1
        if hi <= lo:
            return list(range(max(cls_offset, 0), T))[:n_pos] or [0]
        pos = np.unique(np.round(np.linspace(lo, hi, n_pos)).astype(int)).tolist()
        return pos
    H, W = layout.H, layout.W
    k = int(math.ceil(math.sqrt(n_pos)))
    hs = np.round(np.linspace(0.2 * (H - 1), 0.8 * (H - 1), k)).astype(int)
    ws = np.round(np.linspace(0.2 * (W - 1), 0.8 * (W - 1), k)).astype(int)
    grid = [int(h * W + w) for h in hs for w in ws]
    grid = list(dict.fromkeys(grid))
    return grid[:n_pos]


# ---------------------------------------------------------------- Jacobian products
def _flat_norm(x):
    return x.flatten(1).norm(dim=1).view(-1, *([1] * (x.dim() - 1)))


def jvp_func(f, x, v):
    return torch.func.jvp(f, (x,), (v,))[1]


def jvp_double_backward(f, x, v):
    x = x.detach().requires_grad_(True)
    y = f(x)
    u = torch.zeros_like(y, requires_grad=True)
    g = torch.autograd.grad(y, x, grad_outputs=u, create_graph=True)[0]
    return torch.autograd.grad(g, u, grad_outputs=v.to(g.dtype))[0].detach()


def jvp_fd(f, x, v, eps=1e-3):
    with torch.no_grad():
        s = _flat_norm(x).clamp_min(1e-8) * eps
        return (f(x + s * v) - f(x)) / s


def vjp(f, x, u):
    x = x.detach().requires_grad_(True)
    y = f(x)
    return torch.autograd.grad(y, x, grad_outputs=u.to(y.dtype))[0].detach()


JVP_METHODS = [("torch.func.jvp", jvp_func), ("double-backward", jvp_double_backward), ("finite-difference", jvp_fd)]


def sigma1_power_iteration(f, x0, n_restarts, iters, tol, jvp_state):
    """Batched power iteration on J^T J. x0: (B0, ...) base points; returns (sigma[B0], n_iter).
    jvp_state is a dict caching which JVP implementation works for this model."""
    B0 = x0.shape[0]
    x = x0.repeat_interleave(n_restarts, dim=0)
    g = torch.Generator(device=x.device).manual_seed(SEED)
    v = torch.randn(x.shape, generator=g, device=x.device, dtype=torch.float32)
    v = v / _flat_norm(v)
    sigma = torch.zeros(x.shape[0], device=x.device)
    n_done = 0
    methods = jvp_state.get("methods", JVP_METHODS)
    for it in range(iters):
        Jv = None
        start = jvp_state.get("idx", 0)
        for k in range(start, len(methods)):
            name, fn = methods[k]
            try:
                Jv = fn(f, x, v).float()
                if not torch.isfinite(Jv).all():
                    raise RuntimeError("non-finite JVP")
                if k != start:
                    log(f"    JVP via {name} (earlier methods failed)")
                jvp_state["idx"] = k
                jvp_state["name"] = name
                break
            except Exception as e:
                jvp_state.setdefault("errors", []).append(f"{name}: {type(e).__name__}: {str(e)[:120]}")
                Jv = None
        if Jv is None:
            if jvp_state.get("tolerate_nonfinite"):
                jvp_state["n_nonfinite"] = jvp_state.get("n_nonfinite", 0) + 1
                return torch.full((B0,), float("nan"), device=x.device), it
            raise RuntimeError("all JVP methods failed: " + " | ".join(jvp_state.get("errors", [])))
        s_new = _flat_norm(Jv).flatten()
        u = Jv / _flat_norm(Jv).clamp_min(1e-30)
        JTu = vjp(f, x, u).float()
        v = JTu / _flat_norm(JTu).clamp_min(1e-30)
        n_done = it + 1
        if it >= 4:
            rel = ((s_new - sigma).abs() / s_new.clamp_min(1e-30)).max().item()
            sigma = s_new
            if rel < tol:
                break
        else:
            sigma = s_new
    sigma = sigma.view(B0, n_restarts).max(dim=1).values
    return sigma, n_done


def rho_power_iteration(f, x0, n_restarts, iters, tol, jvp_state):
    """Power iteration on J itself: v <- J v / |J v|. Converges to the spectral radius |lambda_max|
    when the dominant eigenvalue is real; with a complex dominant pair the iterate oscillates, so
    the value reported is the median of the last five |J v| per restart, max over restarts.
    This is the quantity the earlier finite-difference variant estimated."""
    B0 = x0.shape[0]
    x = x0.repeat_interleave(n_restarts, dim=0)
    g = torch.Generator(device=x.device).manual_seed(SEED)
    v = torch.randn(x.shape, generator=g, device=x.device, dtype=torch.float32)
    v = v / _flat_norm(v)
    methods = jvp_state.get("methods", JVP_METHODS)
    hist = []
    n_done = 0
    for it in range(iters):
        Jv = None
        start = jvp_state.get("idx", 0)
        for k in range(start, len(methods)):
            name, fn = methods[k]
            try:
                Jv = fn(f, x, v).float()
                if not torch.isfinite(Jv).all():
                    raise RuntimeError("non-finite JVP")
                jvp_state["idx"] = k; jvp_state["name"] = name
                break
            except Exception as e:
                jvp_state.setdefault("errors", []).append(f"{name}: {type(e).__name__}: {str(e)[:120]}")
                Jv = None
        if Jv is None:
            if jvp_state.get("tolerate_nonfinite"):
                jvp_state["n_nonfinite"] = jvp_state.get("n_nonfinite", 0) + 1
                return torch.full((B0,), float("nan"), device=x.device), it
            raise RuntimeError("all JVP methods failed: " + " | ".join(jvp_state.get("errors", [])))
        r = _flat_norm(Jv).flatten()
        hist.append(r)
        v = Jv / _flat_norm(Jv).clamp_min(1e-30)
        n_done = it + 1
        if it >= 6:
            last = torch.stack(hist[-3:])
            if ((last.max(0).values - last.min(0).values) / last.mean(0).clamp_min(1e-30)).max().item() < tol:
                break
    tail = torch.stack(hist[-5:]) if len(hist) >= 5 else torch.stack(hist)
    rho = tail.median(dim=0).values.view(B0, n_restarts).max(dim=1).values
    return rho, n_done


# ---------------------------------------------------------------- the maps
def make_f_incontext(layer_call, hidden0, layout, positions, branch, model_dtype):
    """f: (B, d) -> (B, d) with B a multiple of len(positions); row i perturbs position
    positions[i // rep] (rows are ordered position-major, matching repeat_interleave)."""
    P = len(positions)
    base1 = hidden0.float()
    M1 = layout.mask(positions, hidden0.device)
    cache = {}

    def f(xb):
        B = xb.shape[0]
        rep = B // P
        if rep not in cache:
            pos_rep = [p for p in positions for _ in range(rep)]
            cache[rep] = (base1.expand(B, *base1.shape[1:]), M1.repeat_interleave(rep, dim=0), pos_rep)
        base, M, pos_rep = cache[rep]
        X = base * (1.0 - M) + layout.broadcast(xb) * M
        Y = layer_call(X.to(model_dtype))
        y = layout.gather(Y, pos_rep).float()
        return y - xb if branch else y
    return f


def make_f_fullseq(layer_call, branch, model_dtype):
    def f(X):
        Y = layer_call(X.to(model_dtype)).float()
        return Y - X if branch else Y
    return f


def _slice_to_position(obj, t, T):
    """Slice every tensor dim of size T down to [t:t+1] (masks, rotary tables, position ids)."""
    if torch.is_tensor(obj):
        if obj.dim() == 0:
            return obj
        out = obj
        for dim in range(obj.dim()):
            if out.shape[dim] == T and T > 1:
                out = out.narrow(dim, t, 1)
        return out
    if isinstance(obj, (tuple, list)):
        return type(obj)(_slice_to_position(o, t, T) for o in obj)
    if isinstance(obj, dict):
        return {k: _slice_to_position(v, t, T) for k, v in obj.items()}
    return obj


def make_f_isolated(layer, rest, kw, layout, t, branch, model_dtype):
    rest_t = _slice_to_position(rest, t, layout.T)
    kw_t = _slice_to_position(kw, t, layout.T)
    call = make_layer_call(layer, rest_t, kw_t)

    def f(xb):                      # (B, d) -> (B, d); each row is a length-1 sequence
        Y = call(xb[:, None, :].to(model_dtype))
        y = Y[:, 0, :].float()
        return y - xb if branch else y
    return f


# ---------------------------------------------------------------- profile
def profile_model(b, context, inputs_kind, branch, args, inputs=None):
    """Returns the per-model result dict for one protocol."""
    arch = b["arch"]
    if branch and arch not in BRANCH_OK:
        raise ValueError(f"--branch is defined for pre-norm residual blocks only ({BRANCH_OK}); {arch} is not")
    if context == "isolated" and b["kind"] != "text":
        raise ValueError("isolated context is defined for sequence models only")
    layers = b["layers"]
    L = len(layers)
    if inputs is None:
        inputs = prepare_inputs(b, inputs_kind, args.n_inputs)
    model_dtype = b["dtype"]
    reduced = model_dtype != torch.float32 or getattr(args, "force_fd", False)
    if reduced:
        # Forward-mode AD is not reliable in half precision, so the reduced-precision comparison
        # uses the finite-difference protocol of the April float16 note (relative step fd_eps)
        # with the block run in the reduced dtype; non-finite outputs are recorded, not fatal.
        from functools import partial
        eps = getattr(args, "fd_eps", 1e-3)
        jvp_state = {"methods": [(f"finite-difference eps={eps:g} ({str(model_dtype).split('.')[-1]})", partial(jvp_fd, eps=eps))],
                     "tolerate_nonfinite": True, "fd_eps": eps}
        log(f"    reduced precision ({str(model_dtype).split('.')[-1]}): finite-difference JVP, relative step {eps:g}")
    else:
        jvp_state = {}
    use_rho = getattr(args, "estimator", "sigma1") == "rho"
    iterate = rho_power_iteration if use_rho else sigma1_power_iteration
    capture_diag = []
    samples = [[] for _ in range(L)]        # per layer: list over (input, position) or (input,) for fullseq
    iters_used = []
    positions_log = []
    t0 = time.time()
    for ii, inp in enumerate(inputs):
        captured = capture_layer_inputs(b, inp)
        with torch.no_grad():
            hmax = [float(c[0].float().abs().max()) for c in captured]
            hfin = [bool(torch.isfinite(c[0]).all()) for c in captured]
        capture_diag.append({"max_abs_hidden": hmax, "finite": hfin})
        if not all(hfin):
            log(f"    input {ii + 1}: non-finite captured hidden state at blocks {[i for i, ok in enumerate(hfin) if not ok]}")
        elif ii == 0:
            log(f"    captured hidden states: max |h| {max(hmax):.3g} (block {int(np.argmax(hmax))}), all finite")
        for li in range(L):
            hidden0, rest, kw = captured[li]
            layout = Layout(hidden0)
            layer_call = make_layer_call(layers[li], rest, kw)
            if context == "fullseq":
                f = make_f_fullseq(layer_call, branch, model_dtype)
                x0 = hidden0.float()
                sig, n_it = iterate(f, x0, args.restarts, args.iters, PI_TOL, jvp_state)
                samples[li].append(float(sig[0]))
                iters_used.append(n_it)
                continue
            positions = choose_positions(layout, args.n_positions, b.get("cls_offset", 0), inp.get("valid_T"))
            if ii == 0 and li in (0, L - 1):
                positions_log.append((li, tuple(hidden0.shape), positions))
            if context == "isolated":
                for t in positions:
                    f = make_f_isolated(layers[li], rest, kw, layout, t, branch, model_dtype)
                    x0 = layout.get(hidden0.float(), [t])
                    sig, n_it = iterate(f, x0, args.restarts, args.iters, PI_TOL, jvp_state)
                    samples[li].append(float(sig[0]))
                    iters_used.append(n_it)
                continue
            # in-context: all positions in one batch, falling back to one position at a time
            x0 = layout.get(hidden0.float(), positions)
            try:
                f = make_f_incontext(layer_call, hidden0, layout, positions, branch, model_dtype)
                sig, n_it = iterate(f, x0, args.restarts, args.iters, PI_TOL, jvp_state)
                samples[li].extend([float(s) for s in sig])
                iters_used.append(n_it)
            except Exception as e:
                if li == 0 and ii == 0:
                    log(f"    batched positions failed ({type(e).__name__}: {str(e)[:100]}); one position at a time")
                jvp_state.pop("idx", None)
                for j, t in enumerate(positions):
                    f = make_f_incontext(layer_call, hidden0, layout, [t], branch, model_dtype)
                    sig, n_it = iterate(f, x0[j:j + 1], args.restarts, args.iters, PI_TOL, jvp_state)
                    samples[li].append(float(sig[0]))
                    iters_used.append(n_it)
        del captured
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        done = [np.mean(s) for s in samples]
        log(f"    input {ii + 1}/{len(inputs)} done ({time.time() - t0:.0f} s); "
            f"block0 {done[0]:.3g}, mid {done[L // 2]:.3g}, last {done[-1]:.3g}")
    prof = np.array([np.nanmean(s) if np.isfinite(s).any() else np.nan for s in samples])
    sd = np.array([np.nanstd(s) if np.isfinite(s).any() else np.nan for s in samples])
    med = np.array([np.nanmedian(s) if np.isfinite(s).any() else np.nan for s in samples])
    if jvp_state.get("n_nonfinite"):
        log(f"    non-finite JVP in {jvp_state['n_nonfinite']} probe batches; affected blocks: "
            f"{[i for i, s in enumerate(samples) if not np.isfinite(s).all()]}")
    stats = profile_stats(prof)
    ci = bootstrap_R_ci(samples)
    res = dict(
        hf_id=b["hf_id"], label=b["cfg"]["label"], arch=arch, family=b["cfg"]["family"],
        params_M=b["cfg"]["params_M"], in_april_panel=b["cfg"]["april"], n_layers=L,
        version=VERSION, protocol=protocol_tag(context, inputs_kind, branch, str(model_dtype).split(".")[-1], getattr(args, "force_fd", False),
                                               getattr(args, "estimator", "sigma1")),
        estimator=(("rho: power iteration on J" if use_rho else "sigma1: power iteration on J^T J")
                   + (" via finite differences" if reduced else " via AD (jvp/vjp)")),
        context=context, inputs=inputs_kind, branch=branch, dtype=str(model_dtype).split(".")[-1],
        definition="sigma_1 of d h_{l+1,t}/d h_{l,t}, other positions held fixed" if context == "incontext" else
                   ("sigma_1 of the block Jacobian for a length-1 sequence at position t" if context == "isolated" else
                    "sigma_1 of the whole-sequence block Jacobian"),
        jvp_method=jvp_state.get("name"), jvp_errors=jvp_state.get("errors", [])[:5],
        fd_eps=jvp_state.get("fd_eps"), n_nonfinite_probe_batches=jvp_state.get("n_nonfinite", 0),
        capture_diagnostics=capture_diag,
        n_inputs=len(inputs), n_positions=args.n_positions, n_restarts=args.restarts, pi_iters=args.iters,
        pi_tol=PI_TOL, mean_iters=float(np.mean(iters_used)) if iters_used else None,
        converged_frac=float(np.mean([n < args.iters for n in iters_used])) if iters_used else None,
        input_desc=b.get("input_desc"), positions_example=positions_log,
        sigma1_profile=prof.tolist(), sigma1_sd=sd.tolist(), sigma1_median=med.tolist(),
        sigma1_samples=[list(map(float, s)) for s in samples],
        **stats, R_ex0_thirds_ci95=ci,
        time_s=time.time() - t0, torch=torch.__version__, transformers=_tf_version(),
        gpu=torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
        date=datetime.datetime.now().isoformat(timespec="seconds"),
    )
    return res


def _tf_version():
    try:
        import transformers
        return transformers.__version__
    except Exception:
        return None


# ---------------------------------------------------------------- statistics
def R_thirds(profile, exclude0=True):
    p = np.asarray(profile, dtype=float)
    p = p[1:] if exclude0 else p
    n = len(p)
    if n < 3:
        return float("nan"), float("nan")
    t = n // 3
    early, mid, late = p[:t], p[t:n - t], p[n - t:]
    edge = np.concatenate([early, late]).mean()
    R = mid.mean() / edge if edge > 0 else float("nan")
    uptick = late.mean() / mid.mean() if mid.mean() > 0 else float("nan")
    return float(R), float(uptick)


def R_vertex(profile):
    """May-census R_ex0: waist median / edge median about the log-quadratic vertex, block 0 excluded."""
    prof = np.asarray(profile, dtype=float)
    n = len(prof)
    if n < 3:
        return float("nan")
    d = np.linspace(0, 1, n)
    X = np.column_stack([d ** 2, d, np.ones(n)])
    beta = np.linalg.lstsq(X, np.log(prof + 1e-12), rcond=None)[0]
    a = beta[0]
    d_star = np.clip(-beta[1] / (2 * a), 0.0, 1.0) if a > 0 else 0.5
    waist = (np.abs(d - d_star) <= 0.20) & (np.arange(n) > 0)
    edge = ((d <= 0.15) | (d >= 0.85)) & (np.arange(n) > 0)
    if waist.sum() == 0 or edge.sum() == 0:
        return float("nan")
    e = np.median(prof[edge])
    return float(np.median(prof[waist]) / e) if e > 0 else float("nan")


def contrast_and_curvature(profile):
    """Census classifier geometry (d3g v5): quadratic coefficient a on [0,1] and
    spectral contrast C = (edge - waist) / edge with 3-block edge means and a 3-block waist."""
    s1 = np.asarray(profile, dtype=float)
    if len(s1) < 4:
        return float("nan"), float("nan")
    x = np.linspace(0, 1, len(s1))
    a = float(np.polyfit(x, s1, 2)[0])
    mid = len(s1) // 2
    edge = (np.mean(s1[:3]) + np.mean(s1[-3:])) / 2
    waist = np.mean(s1[mid - 1:mid + 2])
    C = float((edge - waist) / edge) if edge > 0 else float("nan")
    return C, a


def profile_stats(prof):
    R, up = R_thirds(prof, exclude0=True)
    R_incl0, up_incl0 = R_thirds(prof, exclude0=False)
    C_ex0, a_ex0 = contrast_and_curvature(prof[1:])
    C_incl0, a_incl0 = contrast_and_curvature(prof)
    return dict(R_ex0_thirds=R, late_uptick=up, hourglass=bool(R < 0.80 and up > 1.0) if np.isfinite(R) else None,
                R_thirds_incl0=R_incl0, R_ex0_vertex=R_vertex(prof),
                contrast_C_ex0=C_ex0, quad_a_ex0=a_ex0, contrast_C_incl0=C_incl0, quad_a_incl0=a_incl0,
                block0_over_block1=float(prof[0] / prof[1]) if len(prof) > 1 and prof[1] > 0 else None)


def bootstrap_R_ci(samples, n_boot=N_BOOT):
    """95% CI of R_ex0_thirds, resampling the (input, position) samples with replacement."""
    L = len(samples)
    n = min(len(s) for s in samples)
    if n < 2:
        return [None, None]
    arr = np.array([s[:n] for s in samples], dtype=float)          # (L, n)
    rng = np.random.default_rng(SEED)
    Rs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        with np.errstate(all="ignore"):
            prof = np.nanmean(arr[:, idx], axis=1)
        Rs.append(R_thirds(prof)[0])
    Rs = np.array([r for r in Rs if np.isfinite(r)])
    if len(Rs) == 0:
        return [None, None]
    return [float(np.percentile(Rs, 2.5)), float(np.percentile(Rs, 97.5))]


# ---------------------------------------------------------------- io
def result_path(results_dir, tag, hf_id):
    return os.path.join(results_dir, tag, f"sigma1_{safe_name(hf_id)}.json")


def write_summary(results_dir, tag):
    d = os.path.join(results_dir, tag)
    rows = []
    for fn in sorted(os.listdir(d)):
        if fn.startswith("sigma1_") and fn.endswith(".json"):
            r = json.load(open(os.path.join(d, fn)))
            rows.append({k: r.get(k) for k in (
                "label", "hf_id", "family", "arch", "n_layers", "params_M", "in_april_panel", "R_ex0_thirds",
                "R_ex0_thirds_ci95", "late_uptick", "hourglass", "R_ex0_vertex", "contrast_C_ex0", "quad_a_ex0",
                "contrast_C_incl0", "quad_a_incl0", "block0_over_block1", "jvp_method", "converged_frac",
                "mean_iters", "time_s", "protocol")})
    if not rows:
        return
    for r in rows:
        for k in ("R_ex0_thirds", "late_uptick", "R_ex0_vertex", "contrast_C_ex0", "quad_a_ex0"):
            if r.get(k) is None:
                r[k] = float("nan")
    rows.sort(key=lambda r: (not r["in_april_panel"], r["family"], r["params_M"] or 0))
    json.dump(rows, open(os.path.join(d, "survey_summary.json"), "w"), indent=2)
    import csv
    with open(os.path.join(d, "survey_summary.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        for r in rows:
            rr = dict(r)
            rr["R_ex0_thirds_ci95"] = "" if not r["R_ex0_thirds_ci95"] or r["R_ex0_thirds_ci95"][0] is None else \
                f"{r['R_ex0_thirds_ci95'][0]:.3f}-{r['R_ex0_thirds_ci95'][1]:.3f}"
            w.writerow(rr)
    with open(os.path.join(d, "survey_summary.md"), "w") as f:
        f.write(f"Protocol {tag}\n\n| Model | Family | Blocks | R_ex0 | 95% CI | uptick | HG | R_vertex | C_ex0 | a_ex0 |\n|---|---|---|---|---|---|---|---|---|---|\n")
        for r in rows:
            ci = r["R_ex0_thirds_ci95"]
            ci_s = f"{ci[0]:.2f}-{ci[1]:.2f}" if ci and ci[0] is not None else ""
            f.write(f"| {r['label']} | {r['family']} | {r['n_layers']} | {r['R_ex0_thirds']:.3f} | {ci_s} | "
                    f"{r['late_uptick']:.2f} | {'yes' if r['hourglass'] else 'no'} | {r['R_ex0_vertex']:.3f} | "
                    f"{r['contrast_C_ex0']:.3f} | {r['quad_a_ex0']:.3f} |\n")
    april = [r for r in rows if r["in_april_panel"]]
    log(f"\n  {tag}: {len(rows)} models; April panel {sum(bool(r['hourglass']) for r in april)}/{len(april)} hourglass-positive;"
        f" all {sum(bool(r['hourglass']) for r in rows)}/{len(rows)}")
    log(f"  {'model':26s} {'R_ex0':>6s} {'uptick':>6s} {'HG':>3s} {'C_ex0':>6s}")
    for r in rows:
        log(f"  {r['label']:26s} {r['R_ex0_thirds']:6.3f} {r['late_uptick']:6.2f} {'yes' if r['hourglass'] else 'no':>3s} {r['contrast_C_ex0']:6.3f}")


# ---------------------------------------------------------------- main
def run_validation(cfg, args):
    """Exact token-local Jacobians at five blocks x three positions x two natural inputs:
    exact sigma_1 (SVD), exact rho (eigenvalues), the sigma_1 and rho power-iteration estimates,
    sigma_1 of J - I, and a 64-direction random-probe estimate (max |J v| over random unit v)."""
    out_dir = os.path.join(args.results_dir, "validation")
    path = os.path.join(out_dir, f"validation_{safe_name(cfg['hf_id'])}.json")
    if os.path.exists(path) and not args.no_skip:
        log(f"  skip validation {cfg['hf_id']} (exists)"); return
    b = load_bundle(cfg, "float32")
    if b["kind"] != "text":
        log("  validation mode is written for sequence models"); return
    inputs = prepare_inputs(b, "natural", 2)
    L = len(b["layers"])
    blocks = sorted(set([0, L // 4, L // 2, (3 * L) // 4, L - 1]))
    rows = []
    t0 = time.time()
    for ii, inp in enumerate(inputs):
        captured = capture_layer_inputs(b, inp)
        for li in blocks:
            hidden0, rest, kw = captured[li]
            layout = Layout(hidden0)
            call = make_layer_call(b["layers"][li], rest, kw)
            positions = choose_positions(layout, 3, 0, inp.get("valid_T"))
            for t in positions:
                f = make_f_incontext(call, hidden0, layout, [t], False, torch.float32)
                x0 = layout.get(hidden0.float(), [t])
                d = x0.shape[1]
                try:
                    J = torch.autograd.functional.jacobian(f, x0, vectorize=True)
                except Exception:
                    J = torch.autograd.functional.jacobian(f, x0)
                J = J.reshape(d, d).detach()
                sv = torch.linalg.svdvals(J)
                ev = torch.linalg.eigvals(J).abs()
                s1_exact, s2_exact, rho_exact = sv[0].item(), sv[1].item(), ev.max().item()
                s1_branch = torch.linalg.svdvals(J - torch.eye(d, device=J.device))[0].item()
                s1_est, n1 = sigma1_power_iteration(f, x0, args.restarts, args.iters, PI_TOL, {})
                rho_est, n2 = rho_power_iteration(f, x0, args.restarts, args.iters, PI_TOL, {})
                g = torch.Generator(device=J.device).manual_seed(SEED)
                V = torch.randn(64, d, generator=g, device=J.device); V = V / V.norm(dim=1, keepdim=True)
                probe = (V @ J.T).norm(dim=1).max().item()
                rows.append(dict(input=ii, block=li, position=int(t), d=d, sigma1_exact=s1_exact, sigma2_exact=s2_exact,
                                 rho_exact=rho_exact, sigma1_branch_exact=s1_branch, sigma1_est=float(s1_est[0]),
                                 sigma1_est_iters=n1, rho_est=float(rho_est[0]), rho_est_iters=n2, random_probe_64=probe))
                log(f"    input {ii} block {li:3d} pos {t:3d}: sigma1 {s1_exact:8.3f} (est {float(s1_est[0]):8.3f}, {n1} it)  "
                    f"rho {rho_exact:7.3f} (est {float(rho_est[0]):7.3f})  sigma1(J-I) {s1_branch:7.3f}  probes64 {probe:7.3f}  "
                    f"sigma1/rho {s1_exact / max(rho_exact, 1e-9):5.2f}")
        del captured
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    s1e = np.array([r["sigma1_exact"] for r in rows]); s1p = np.array([r["sigma1_est"] for r in rows])
    rhe = np.array([r["rho_exact"] for r in rows]); rhp = np.array([r["rho_est"] for r in rows]); pr = np.array([r["random_probe_64"] for r in rows])
    summary = dict(hf_id=cfg["hf_id"], label=cfg["label"], n=len(rows), blocks=blocks,
                   sigma1_max_rel_err=float(np.max(np.abs(s1p - s1e) / s1e)), sigma1_pearson_r=float(np.corrcoef(s1e, s1p)[0, 1]),
                   rho_max_rel_err=float(np.max(np.abs(rhp - rhe) / rhe)),
                   probe_underestimate_range=[float(np.min(1 - pr / s1e)), float(np.max(1 - pr / s1e))],
                   sigma1_over_rho_range=[float(np.min(s1e / rhe)), float(np.max(s1e / rhe))],
                   time_s=time.time() - t0, version=VERSION)
    os.makedirs(out_dir, exist_ok=True)
    json.dump(dict(summary=summary, rows=rows), open(path, "w"), indent=1)
    log(f"  validation: sigma1 estimator max rel err {summary['sigma1_max_rel_err']:.2e} (r = {summary['sigma1_pearson_r']:.4f}); "
        f"rho estimator max rel err {summary['rho_max_rel_err']:.2e}; random probes underestimate by "
        f"{100 * summary['probe_underestimate_range'][0]:.0f}-{100 * summary['probe_underestimate_range'][1]:.0f}%; "
        f"sigma1/rho {summary['sigma1_over_rho_range'][0]:.1f}-{summary['sigma1_over_rho_range'][1]:.1f}; saved {path}")
    del b; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def _purge_hub_cache(hf_id):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        d = os.path.join(HF_HUB_CACHE, "models--" + hf_id.replace("/", "--"))
        if os.path.isdir(d):
            shutil.rmtree(d, ignore_errors=True)
            log(f"  purged Hub cache for {hf_id}")
    except Exception as e:
        log(f"  (cache purge failed: {e})")


def _print_smoke_anchors(path):
    p = json.load(open(path))["sigma1_profile"]
    log("  smoke reference (v35 Fig. 1a exact-Jacobian sigma_1 on GPT-2 124M): " +
        ", ".join(f"L{k} {v:.1f} (now {p[k]:.2f})" for k, v in GPT2_REFERENCE.items()))


def run_one(cfg, protocols, args):
    """Load a model once and run every requested protocol on it."""
    todo = []
    if args.validate:
        run_validation(cfg, args)
        return
    for (context, inputs_kind, branch, dtype_name) in protocols:
        tag = protocol_tag(context, inputs_kind, branch, dtype_name, args.force_fd, args.estimator)
        path = result_path(args.results_dir, tag, cfg["hf_id"])
        if os.path.exists(path) and not args.no_skip:
            log(f"  skip {cfg['hf_id']} [{tag}] (exists)")
            if args.smoke and cfg["hf_id"] == "gpt2" and context == "incontext" and not branch:
                _print_smoke_anchors(path)
            continue
        if branch and cfg["arch"] not in BRANCH_OK:
            log(f"  skip {cfg['hf_id']} [{tag}]: branch mode not defined for {cfg['arch']}")
            continue
        if context == "isolated" and cfg["arch"] not in TEXT_ARCHES:
            log(f"  skip {cfg['hf_id']} [{tag}]: isolated context not defined for {cfg['arch']}")
            continue
        todo.append((context, inputs_kind, branch, dtype_name, tag, path))
    if not todo:
        return
    by_dtype = {}
    for item in todo:
        by_dtype.setdefault(item[3], []).append(item)
    for dtype_name, items in by_dtype.items():
        b = None
        try:
            b = load_bundle(cfg, dtype_name)
            input_cache = {}
            for (context, inputs_kind, branch, _, tag, path) in items:
                log(f"  [{tag}]")
                if inputs_kind not in input_cache:
                    input_cache[inputs_kind] = prepare_inputs(b, inputs_kind, args.n_inputs)
                res = profile_model(b, context, inputs_kind, branch, args, inputs=input_cache[inputs_kind])
                os.makedirs(os.path.dirname(path), exist_ok=True)
                json.dump(res, open(path, "w"), indent=1)
                log(f"  saved {path}\n    R_ex0={res['R_ex0_thirds']:.3f} (CI {res['R_ex0_thirds_ci95']}), uptick={res['late_uptick']:.2f}, "
                    f"hourglass={res['hourglass']}, R_vertex={res['R_ex0_vertex']:.3f}, C_ex0={res['contrast_C_ex0']:.3f}, "
                    f"jvp={res['jvp_method']}, {res['time_s']:.0f} s")
                if args.smoke and cfg["hf_id"] == "gpt2" and context == "incontext" and not branch:
                    _print_smoke_anchors(path)
        except Exception as e:
            log(f"  FAILED {cfg['hf_id']} [{dtype_name}]: {type(e).__name__}: {e}")
            traceback.print_exc()
        finally:
            if b is not None:
                del b
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    if args.purge_cache:
        _purge_hub_cache(cfg["hf_id"])


def main(argv=None):
    args = parse_args(argv)
    _resolve_hf_token()
    log(f"{VERSION}; device {DEVICE}; torch {torch.__version__}; transformers {_tf_version()}")
    if DEVICE == "cuda":
        p = torch.cuda.get_device_properties(0)
        a = torch.randn(2, 2, device="cuda") @ torch.randn(2, 2, device="cuda")
        log(f"GPU: {p.name} ({p.total_memory / 1e9:.0f} GB, sm_{p.major}{p.minor}); TF32 conv={torch.backends.cudnn.allow_tf32} "
            f"matmul={torch.backends.cuda.matmul.allow_tf32}; HF token: {'found' if os.environ.get('HF_TOKEN') else 'none'}")
    log(f"settings: inputs {args.n_inputs} x positions {args.n_positions}, restarts {args.restarts}, iters {args.iters}, tol {PI_TOL}")

    if args.protocol_panel:
        models = args.models or PROTOCOL_PANEL_MODELS
        protocols = [(c, i, False, "float32") for (c, i) in PROTOCOL_PANEL]
    else:
        if args.models:
            models = args.models
        elif args.smoke:
            models = ["gpt2"]
        else:
            models = [r[0] for r in REGISTRY if (args.panel == "union" or r[5])]
        protocols = [(args.context, args.inputs, args.branch, args.dtype)]
    unknown = [m for m in models if m not in CFG]
    if unknown:
        raise SystemExit(f"unknown model ids (add to REGISTRY): {unknown}")
    order = {r[0]: i for i, r in enumerate(REGISTRY)}
    models = sorted(models, key=lambda m: (CFG[m]["params_M"], order[m]))
    log(f"models ({len(models)}): {models}")
    log(f"protocols: {[protocol_tag(*p, args.force_fd, args.estimator) for p in protocols]}" + (" (validation mode)" if args.validate else ""))
    os.makedirs(args.results_dir, exist_ok=True)
    t0 = time.time()
    for i, m in enumerate(models):
        log(f"\n{'=' * 78}\n[{i + 1}/{len(models)}] {m} ({CFG[m]['arch']}, ~{CFG[m]['params_M']}M)\n{'=' * 78}")
        run_one(CFG[m], protocols, args)
    for p in protocols:
        tag = protocol_tag(*p, args.force_fd, args.estimator)
        if os.path.isdir(os.path.join(args.results_dir, tag)):
            write_summary(args.results_dir, tag)
    log(f"\nTotal {time.time() - t0:.0f} s")
    if args.zip:
        z = shutil.make_archive("survey_results", "zip", args.results_dir)
        log(f"zipped: {z}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/reprofile_v2.py
#!/usr/bin/env python3
"""
Re-profile the topology, residual-alpha and k-gram experiments with the canonical estimator
==========================================================================================
The March scripts (b2_topology_intervention.py, e15_bypass_alpha_sweep.py, d3_ngram_sweep.py)
profiled with a jvp-only iteration on the whole-sequence block Jacobian at uniform-random token
ids and without the attention mask, which estimates the spectral radius, not sigma_1. Their
checkpoints were not kept. This script retrains the same models (same architecture, corpus,
optimiser, schedule and seeds as the originals) and measures

  geometry     token-local sigma_1 per block by survey_sigma1_v2 (J^T J power iteration, causal
               mask, 5 inputs x 8 positions, 4 restarts) on natural WikiText text AND on
               in-distribution text; R_ex0, two-arm rule, original C/U-fit; at init, 5k, 10k
  sensitivity  branch rotation (attention and feed-forward outputs before the residual add, the
               d1/k-gram/census intervention) by fixed region (early 0-3, waist 4-7, late 8-11) at
               doses 0.25/0.5/1 (linear interpolation as in d3) and by the original fitted-waist
               regions; per-block branch rotation; per-block identity skip; skip by region
  early exit   final norm + tied head applied at every block output

Experiments (REPRO_EXP or --exp):
  topology  6 conditions x 3 seeds (baseline, no_residual, gated_middle, highway, multi_highway,
            unet_skip), WikiText-103 20k-sequence corpus, AdamW wd 0.1 betas (0.9, 0.95), cosine
  alpha     alpha in {0, 0.25, 0.5, 0.75, 1} x 3 seeds, same training as topology
  kgram     k in {1, 2, 3, 4, 5, 8} x 5 seeds, k-gram back-off sources fitted to the same corpus
            (top-200 successors per context), AdamW wd 0.01, cosine; held-out sequences for eval

Every run writes one JSON atomically, resume-safe; fp16 checkpoints in <OUT>/ckpt; results zip
written at the end. Needs survey_sigma1_v2.py beside it.
"""

import os, sys, json, time, math, gc, datetime, argparse, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import survey_sigma1_v2 as sv

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path(os.environ.get("REPRO_OUT", "reprofile_v2"))
SEED_BASE = 42
VOCAB_SIZE = 50257
CFG = dict(n_layers=12, d_model=512, n_heads=8, d_ff=2048, seq_len=128, batch_size=16, n_steps=10000, lr=3e-4, warmup_steps=500)
PROFILE_AT = [5000]
EARLY, WAIST, LATE = [0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]
DOSES = [0.25, 0.5, 1.0]
N_EVAL_BATCHES, EVAL_BATCH = 50, 16
N_BOOT = 500
TOPOLOGIES = ["baseline", "no_residual", "gated_middle", "highway", "multi_highway", "unet_skip"]
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
K_VALUES = [1, 2, 3, 4, 5, 8]
N_TRAIN_SEQS, TOP_K_NGRAM = 20000, 200
CONTRAST_WAIST_W, BOUNDARY_CUT = 0.20, 0.15


def log(msg=""):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


# ================================================================ models (verbatim from the March scripts)
class B2Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, residual_scale=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self._gate_param = None
        if residual_scale is None or (not isinstance(residual_scale, nn.Parameter) and residual_scale == 1.0):
            self._residual_mode = "standard"
        elif not isinstance(residual_scale, nn.Parameter) and residual_scale == 0.0:
            self._residual_mode = "none"
        elif isinstance(residual_scale, nn.Parameter):
            self._residual_mode = "gated"; self._gate_param = residual_scale
        else:
            self._residual_mode = "fixed"; self.register_buffer("_fixed_scale", torch.tensor(float(residual_scale)))

    def forward(self, x, attn_mask=None):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask)
        if self._residual_mode == "none": x2 = attn_out
        elif self._residual_mode == "gated": x2 = torch.sigmoid(self._gate_param) * x + attn_out
        elif self._residual_mode == "fixed": x2 = self._fixed_scale * x + attn_out
        else: x2 = x + attn_out
        h = self.ln2(x2)
        mlp_out = self.mlp(h)
        if self._residual_mode == "none": return mlp_out
        elif self._residual_mode == "gated": return torch.sigmoid(self._gate_param) * x2 + mlp_out
        elif self._residual_mode == "fixed": return self._fixed_scale * x2 + mlp_out
        return x2 + mlp_out


class B2Transformer(nn.Module):
    def __init__(self, topology="baseline"):
        super().__init__()
        cfg = CFG; self.cfg = cfg; self.topology = topology
        d, n_layers = cfg["d_model"], cfg["n_layers"]
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d); self.pos_emb = nn.Embedding(cfg["seq_len"], d); self.drop = nn.Dropout(0.1)
        blocks = []; self.gate_params = nn.ParameterList()
        for i in range(n_layers):
            if topology == "no_residual":
                block = B2Block(d, cfg["n_heads"], cfg["d_ff"], residual_scale=0.0)
            elif topology == "gated_middle" and 4 <= i <= 8:
                gate = nn.Parameter(torch.tensor(3.0)); self.gate_params.append(gate)
                block = B2Block(d, cfg["n_heads"], cfg["d_ff"], residual_scale=gate)
            else:
                block = B2Block(d, cfg["n_heads"], cfg["d_ff"])
            blocks.append(block)
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB_SIZE, bias=False); self.head.weight = self.tok_emb.weight
        if topology == "highway":
            self.highway_proj = nn.Linear(d, d, bias=False); nn.init.zeros_(self.highway_proj.weight)
        if topology == "multi_highway":
            self.mh_projs = nn.ModuleDict({k: nn.Linear(d, d, bias=False) for k in ("1_7", "3_9", "5_11")})
            for p in self.mh_projs.values(): nn.init.zeros_(p.weight)
        if topology == "unet_skip":
            self.unet_projs = nn.ModuleDict({k: nn.Linear(d, d, bias=False) for k in ("1_10", "2_9", "3_8")})
            for p in self.unet_projs.values(): nn.init.zeros_(p.weight)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            skip_projs = set()
            if hasattr(self, "highway_proj"): skip_projs.add(self.highway_proj)
            if hasattr(self, "mh_projs"): skip_projs.update(self.mh_projs.values())
            if hasattr(self, "unet_projs"): skip_projs.update(self.unet_projs.values())
            if module in skip_projs: return
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding): nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.LayerNorm): nn.init.ones_(module.weight); nn.init.zeros_(module.bias)

    def forward(self, input_ids, attn_mask=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(pos))
        if self.topology in ("highway", "multi_highway", "unet_skip"):
            lo = {}
            for i, block in enumerate(self.blocks):
                if self.topology == "highway":
                    if i == 9 and 2 in lo: x = x + self.highway_proj(lo[2])
                elif self.topology == "multi_highway":
                    if i == 7 and 1 in lo: x = x + self.mh_projs["1_7"](lo[1])
                    if i == 9 and 3 in lo: x = x + self.mh_projs["3_9"](lo[3])
                    if i == 11 and 5 in lo: x = x + self.mh_projs["5_11"](lo[5])
                elif self.topology == "unet_skip":
                    if i == 8 and 3 in lo: x = x + self.unet_projs["3_8"](lo[3])
                    if i == 9 and 2 in lo: x = x + self.unet_projs["2_9"](lo[2])
                    if i == 10 and 1 in lo: x = x + self.unet_projs["1_10"](lo[1])
                if self.topology == "highway" and i == 2: lo[2] = x.clone()
                elif self.topology == "multi_highway" and i in (1, 3, 5): lo[i] = x.clone()
                elif self.topology == "unet_skip" and i in (1, 2, 3): lo[i] = x.clone()
                x = block(x, attn_mask=attn_mask)
        else:
            for block in self.blocks:
                x = block(x, attn_mask=attn_mask)
        return self.head(self.ln_f(x))

    def get_gate_values(self):
        return [float(torch.sigmoid(g).item()) for g in self.gate_params] if self.topology == "gated_middle" else None


class AlphaBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, alpha=1.0):
        super().__init__()
        self.alpha = alpha
        self.ln1 = nn.LayerNorm(d_model); self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model); self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, attn_mask=None):
        h = self.ln1(x); attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask); x2 = self.alpha * x + attn_out
        h = self.ln2(x2); return self.alpha * x2 + self.mlp(h)


class AlphaTransformer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        cfg = CFG; self.cfg = cfg; self.alpha = alpha; d = cfg["d_model"]
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d); self.pos_emb = nn.Embedding(cfg["seq_len"], d); self.drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([AlphaBlock(d, cfg["n_heads"], cfg["d_ff"], alpha=alpha) for _ in range(cfg["n_layers"])])
        self.ln_f = nn.LayerNorm(d); self.head = nn.Linear(d, VOCAB_SIZE, bias=False); self.head.weight = self.tok_emb.weight
        self.apply(_init_plain)

    def forward(self, input_ids, attn_mask=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(torch.arange(T, device=input_ids.device).unsqueeze(0)))
        for block in self.blocks: x = block(x, attn_mask=attn_mask)
        return self.head(self.ln_f(x))


def _init_plain(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, std=0.02)
        if module.bias is not None: nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding): nn.init.normal_(module.weight, std=0.02)
    elif isinstance(module, nn.LayerNorm): nn.init.ones_(module.weight); nn.init.zeros_(module.bias)


class D3Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model); self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, attn_mask=None):
        h = self.ln1(x); attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask); x = x + attn_out
        h = self.ln2(x); return x + self.mlp(h)


class D3Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        cfg = CFG; self.cfg = cfg; d = cfg["d_model"]
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d); self.pos_emb = nn.Embedding(cfg["seq_len"], d); self.drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([D3Block(d, cfg["n_heads"], cfg["d_ff"]) for _ in range(cfg["n_layers"])])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(VOCAB_SIZE, d, bias=False); self.head.weight = self.tok_emb.weight
        self.apply(_init_plain)

    def forward(self, input_ids, attn_mask=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(torch.arange(T, device=input_ids.device).unsqueeze(0)))
        for block in self.blocks: x = block(x, attn_mask=attn_mask)
        return F.linear(self.ln_f(x), self.tok_emb.weight)


def head_logits(model, h):
    return F.linear(model.ln_f(h), model.tok_emb.weight)


# ================================================================ data
def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()


def load_wikitext_corpus():
    """The March corpus: first 20,000 WikiText-103 train texts (>100 chars) that tokenise to >= 128
    tokens, truncated to 128; validation from WikiText-2. Cached under ROOT/data."""
    cache = ROOT / "data" / "wikitext_20k_128.pt"
    if cache.exists():
        d = torch.load(cache); log(f"corpus cache: {cache} ({len(d['train'])} train, {len(d['val'])} val)"); return d["train"], d["val"]
    from transformers import AutoTokenizer
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token
    log("tokenising the March corpus (a few minutes the first time)...")
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
    train = []
    for t in ds["text"]:
        if len(t.strip()) <= 100: continue
        ids = tok(t, truncation=True, max_length=CFG["seq_len"], return_tensors="pt").input_ids[0]
        if len(ids) == CFG["seq_len"]: train.append(ids)
        if len(train) >= N_TRAIN_SEQS: break
    vds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
    val = []
    for t in vds["text"]:
        if len(t.strip()) > 50:
            ids = tok(t, truncation=True, max_length=CFG["seq_len"], return_tensors="pt").input_ids[0]
            if len(ids) == CFG["seq_len"]: val.append(ids)
    train, val = torch.stack(train), torch.stack(val)
    cache.parent.mkdir(parents=True, exist_ok=True); torch.save(dict(train=train, val=val), cache)
    log(f"  train {tuple(train.shape)}, val {tuple(val.shape)}")
    return train, val


def build_ngram_models(all_ids):
    """k-gram tables as in d3_ngram_sweep.build_ngram_model: top-200 successors per context, stored as
    (token array, cumulative-probability array) for fast sampling."""
    models = {}
    for k in sorted(set(K_VALUES)):
        if k == 1: models[1] = {}; continue
        c = k - 1
        log(f"  building {k}-gram table (context {c})...")
        counts = defaultdict(lambda: defaultdict(int))
        for i in range(c, len(all_ids)):
            counts[tuple(all_ids[i - c:i])][all_ids[i]] += 1
        table = {}
        for ctx, nxt in counts.items():
            top = sorted(nxt.items(), key=lambda x: -x[1])[:TOP_K_NGRAM]
            tot = sum(v for _, v in top)
            table[ctx] = (np.array([t for t, _ in top], dtype=np.int64), np.cumsum([v / tot for _, v in top]))
        models[k] = table
        log(f"    {len(table)} contexts")
        del counts
    return models


def generate_kgram(k, models, unigram_cum, n_seqs, seq_len, seed):
    """d3_ngram_sweep.generate_kgram_sequences: back-off k -> k-1 -> ... -> unigram; first k-1 tokens
    from the highest-order table that fits."""
    rng = np.random.RandomState(seed)
    out = np.zeros((n_seqs, seq_len), dtype=np.int64)
    U = rng.random_sample((n_seqs, seq_len))
    for s in range(n_seqs):
        seq = []
        for pos in range(seq_len):
            u = U[s, pos]; tok = None
            hi = min(pos + 1, k) if pos < k - 1 else k
            for bk in range(hi, 0, -1):
                if bk == 1:
                    tok = int(np.searchsorted(unigram_cum, u)); break
                c = bk - 1
                if len(seq) >= c and bk in models:
                    ent = models[bk].get(tuple(seq[-c:]))
                    if ent is not None:
                        toks, cum = ent; tok = int(toks[min(np.searchsorted(cum, u), len(toks) - 1)]); break
            if tok is None: tok = int(np.searchsorted(unigram_cum, u))
            seq.append(min(tok, VOCAB_SIZE - 1))
        out[s] = seq
    return torch.from_numpy(out)


def kgram_data(k, seed, corpus, models, unigram_cum):
    cache = ROOT / "data" / f"kgram_k{k}_s{seed}.pt"
    if cache.exists(): return torch.load(cache)
    t0 = time.time()
    train = generate_kgram(k, models, unigram_cum, N_TRAIN_SEQS, CFG["seq_len"], seed=k * 1000 + seed)
    held = generate_kgram(k, models, unigram_cum, N_EVAL_BATCHES * EVAL_BATCH, CFG["seq_len"], seed=k * 1000 + seed + 777)
    d = dict(train=train, held=held); cache.parent.mkdir(parents=True, exist_ok=True); torch.save(d, cache)
    log(f"  generated {k}-gram data seed {seed} ({time.time() - t0:.0f} s)")
    return d


# ================================================================ measurement
def survey_profile(model, mask, ids, n_positions=8, restarts=4, iters=50):
    model.eval()
    # parameters keep requires_grad=True: with it off, nn.MultiheadAttention takes its fused fast path,
    # which forward-mode AD does not support, and the estimator falls back to the slower double-backward JVP
    layers = list(model.blocks)
    b = dict(model=model, layers=layers, dtype=torch.float32, cls_offset=0, forward=lambda m, inp: m(inp["input_ids"], attn_mask=mask))
    L = len(layers); samples = [[] for _ in range(L)]; its = []
    for i in range(ids.shape[0]):
        captured = sv.capture_layer_inputs(b, {"input_ids": ids[i:i + 1].to(DEVICE)})
        for li in range(L):
            hidden0, rest, kw = captured[li]
            layout = sv.Layout(hidden0); call = sv.make_layer_call(layers[li], rest, kw)
            positions = sv.choose_positions(layout, n_positions, 0, None)
            x0 = layout.get(hidden0.float(), positions)
            f = sv.make_f_incontext(call, hidden0, layout, positions, False, torch.float32)
            sig, n_it = sv.sigma1_power_iteration(f, x0, restarts, iters, sv.PI_TOL, {})
            samples[li].extend(float(s) for s in sig); its.append(n_it)
        del captured
    model.train()
    return np.array([np.mean(s) for s in samples]), samples, float(np.mean(its))


def two_arm(prof):
    p = np.asarray(prof, float)[1:]; L = len(p); n = L // 3
    e, m, l = p[:n].mean(), p[n:L - n].mean(), p[L - n:].mean(); R = m / ((e + l) / 2)
    pf = np.asarray(prof, float); nf = len(pf) // 3
    C = float((np.concatenate([pf[:nf], pf[-nf:]]).mean() - pf[nf:-nf].mean()) / pf.mean())
    return dict(R_ex0=float(R), early_over_mid=float(e / m), late_over_mid=float(l / m),
                pooled_rule=bool(R < 0.80 and l > m), two_arm_rule=bool(R < 0.80 and e > m and l > m), contrast_C=C)


def fit_ushape(profile):
    from scipy import stats
    prof = np.array(profile); n = len(prof); lp = np.log(prof + 1e-12); d = np.linspace(0, 1, n)
    X = np.column_stack([d ** 2, d, np.ones(n)]); beta = np.linalg.lstsq(X, lp, rcond=None)[0]; a = beta[0]
    d_star = np.clip(-beta[1] / (2 * a), 0.0, 1.0) if a > 0 else 0.5
    wm = (np.abs(d - d_star) <= CONTRAST_WAIST_W) & (np.arange(n) > 0)
    im, om = d <= BOUNDARY_CUT, d >= 1 - BOUNDARY_CUT
    med = lambda m: float(np.median(lp[m])) if m.sum() > 0 else 0.0
    dE, dT = med(im) - med(wm), med(om) - med(wm)
    waist = [i for i in range(n) if wm[i]]
    w = len(waist)
    early = [i for i in range(1, n) if waist and i < min(waist)][-w:] if w else []
    late = [i for i in range(n) if waist and i > max(waist)][:w] if w else []
    return dict(a=float(a), d_star=float(d_star), delta_E=float(dE), delta_T=float(dT), C_fit=float(min(dE, dT)), waist=waist, early=early, late=late)


def haar(d, seed):
    g = torch.Generator().manual_seed(seed)
    Q, R = torch.linalg.qr(torch.randn(d, d, generator=g))
    return Q * torch.sign(torch.diag(R))[None, :]


class Hooks:
    def __init__(self): self.h = []
    def add(self, x): self.h.append(x)
    def clear(self):
        for x in self.h: x.remove()
        self.h.clear()


def rotate_branches(model, blocks, Qa, Qf, dose=1.0):
    """d3.forward_with_intervention: u <- (1 - dose) u + dose (u Q^T) on attention and MLP outputs."""
    def setup(hk):
        for li in blocks:
            A = Qa[li].to(DEVICE); Fq = Qf[li].to(DEVICE)
            hk.add(model.blocks[li].attn.register_forward_hook(lambda m, a, o, Q=A: ((1 - dose) * o[0] + dose * (o[0] @ Q.T),) + tuple(o[1:])))
            hk.add(model.blocks[li].mlp.register_forward_hook(lambda m, a, o, Q=Fq: (1 - dose) * o + dose * (o @ Q.T)))
    return setup


def skip_blocks(model, blocks):
    def setup(hk):
        for li in blocks:
            hk.add(model.blocks[li].register_forward_hook(lambda m, a, o: a[0]))
    return setup


@torch.no_grad()
def eval_losses(model, data, setup=None):
    model.eval(); model.drop.p = 0.0; hk = Hooks(); mask = causal_mask(CFG["seq_len"], DEVICE)
    try:
        if setup: setup(hk)
        out = []
        for b in range(min(N_EVAL_BATCHES, len(data) // EVAL_BATCH)):
            ids = data[b * EVAL_BATCH:(b + 1) * EVAL_BATCH].to(DEVICE)
            logits = model(ids, attn_mask=mask)
            out.append(F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1)).item())
        return out
    finally:
        hk.clear(); model.drop.p = 0.1; model.train()


def dstat(losses, base):
    a, b = np.array(losses), np.array(base); rng = np.random.default_rng(0); n = len(a)
    boots = [a[i].mean() - b[i].mean() for i in (rng.integers(0, n, n) for _ in range(N_BOOT))]
    return dict(dL=float(a.mean() - b.mean()), ci=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))])


@torch.no_grad()
def early_exit(model, data):
    model.eval(); model.drop.p = 0.0; L = len(model.blocks); mask = causal_mask(CFG["seq_len"], DEVICE)
    outs = [None] * L; hk = Hooks()
    for li in range(L):
        hk.add(model.blocks[li].register_forward_hook(lambda m, a, o, li=li: outs.__setitem__(li, o)))
    tot = np.zeros(L); n = 0
    try:
        for b in range(min(N_EVAL_BATCHES, len(data) // EVAL_BATCH)):
            ids = data[b * EVAL_BATCH:(b + 1) * EVAL_BATCH].to(DEVICE)
            model(ids, attn_mask=mask)
            for li in range(L):
                lg = head_logits(model, outs[li])
                tot[li] += F.cross_entropy(lg[:, :-1].reshape(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1)).item()
            n += 1
    finally:
        hk.clear(); model.drop.p = 0.1; model.train()
    return (tot / max(n, 1)).tolist()


def measure(model, seed, eval_data, probe_nat, probe_dist):
    """Everything measured on one trained model."""
    L = len(model.blocks); out = {}
    mask = causal_mask(CFG["seq_len"], DEVICE)
    prof_nat, samp_nat, it1 = survey_profile(model, mask, probe_nat)
    prof_dist, samp_dist, it2 = survey_profile(model, mask, probe_dist)
    out["profile_natural"], out["profile_indist"] = prof_nat.tolist(), prof_dist.tolist()
    out["samples_natural"], out["samples_indist"] = [[round(x, 4) for x in s] for s in samp_nat], [[round(x, 4) for x in s] for s in samp_dist]
    out["stats_natural"], out["stats_indist"] = two_arm(prof_nat), two_arm(prof_dist)
    out["ufit_indist"] = fit_ushape(prof_dist); out["ufit_natural"] = fit_ushape(prof_nat)
    out["power_iters"] = [it1, it2]
    base = eval_losses(model, eval_data); out["baseline_eval_loss"] = float(np.mean(base))
    Qa = {li: haar(CFG["d_model"], SEED_BASE + li * 100 + seed) for li in range(L)}
    Qf = {li: haar(CFG["d_model"], SEED_BASE + li * 100 + 10 + seed) for li in range(L)}
    regions = {}
    for rn, blk in (("early", EARLY), ("waist", WAIST), ("late", LATE)):
        for dose in DOSES:
            regions[f"{rn}@{dose}"] = dstat(eval_losses(model, eval_data, rotate_branches(model, blk, Qa, Qf, dose)), base)
        regions[rn] = regions[f"{rn}@1.0"]
    out["regions_branch"] = regions
    edge = max(regions["early"]["dL"], regions["late"]["dL"], 1e-9); out["waist_over_edge_branch"] = regions["waist"]["dL"] / edge
    fit = out["ufit_indist"]; fr = {}
    for rn in ("waist", "early", "late"):
        if fit[rn]:
            fr[rn] = dstat(eval_losses(model, eval_data, rotate_branches(model, fit[rn], Qa, Qf, 1.0)), base)
    out["regions_branch_fitted_waist"] = fr
    out["skip_regions"] = {rn: dstat(eval_losses(model, eval_data, skip_blocks(model, blk)), base) for rn, blk in (("early", EARLY), ("waist", WAIST), ("late", LATE))}
    sedge = max(out["skip_regions"]["early"]["dL"], out["skip_regions"]["late"]["dL"], 1e-9); out["skip_waist_over_edge"] = out["skip_regions"]["waist"]["dL"] / sedge
    out["per_block_branch_dL"] = [dstat(eval_losses(model, eval_data, rotate_branches(model, [li], Qa, Qf, 1.0)), base)["dL"] for li in range(L)]
    out["per_block_skip_dL"] = [dstat(eval_losses(model, eval_data, skip_blocks(model, [li])), base)["dL"] for li in range(L)]
    out["early_exit_loss"] = early_exit(model, eval_data)
    return out


# ================================================================ training
def get_lr(step, warm_from_zero):
    w, n, lr = CFG["warmup_steps"], CFG["n_steps"], CFG["lr"]
    if step < w: return lr * step / w
    return lr * 0.5 * (1 + math.cos(math.pi * (step - w) / (n - w)))


def train_model(model, train, seed, exp, probe_cb):
    """topology/alpha: sequential batches over the fixed corpus, AdamW wd 0.1 betas (0.9, 0.95);
    kgram: permuted epochs, AdamW wd 0.01. Cosine schedule with 500 warm-up, clip 1.0, as the originals."""
    if exp == "kgram":
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.1, betas=(0.9, 0.95))
    mask = causal_mask(CFG["seq_len"], DEVICE); bs = CFG["batch_size"]; model.train()
    step, t0, last = 0, time.time(), None
    if exp == "kgram":
        gperm = torch.Generator().manual_seed(SEED_BASE + seed)
        while step < CFG["n_steps"]:
            perm = torch.randperm(len(train), generator=gperm)
            for i in range(0, len(perm) - bs, bs):
                if step >= CFG["n_steps"]: break
                if step in PROFILE_AT: probe_cb(step)
                ids = train[perm[i:i + bs]].to(DEVICE)
                for pg in opt.param_groups: pg["lr"] = get_lr(step, True)
                loss = F.cross_entropy(model(ids, attn_mask=mask)[:, :-1].reshape(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                step += 1; last = loss.item()
                if step % 2500 == 0: log(f"    step {step}/{CFG['n_steps']} loss {last:.3f} ({time.time() - t0:.0f} s)")
    else:
        data_idx = 0
        for step in range(1, CFG["n_steps"] + 1):
            if step - 1 in PROFILE_AT: probe_cb(step - 1)
            for pg in opt.param_groups: pg["lr"] = get_lr(step, True)
            ids = torch.stack([train[(data_idx + i) % len(train)] for i in range(bs)]).to(DEVICE); data_idx += bs
            loss = F.cross_entropy(model(ids, attn_mask=mask)[:, :-1].reshape(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            last = loss.item()
            if step % 2500 == 0: log(f"    step {step}/{CFG['n_steps']} loss {last:.3f} ({time.time() - t0:.0f} s)")
    return last


def run_one(exp, cond, seed, train, eval_data, probe_nat, probe_dist, ckpt_dir):
    t0 = time.time()
    torch.manual_seed(SEED_BASE + seed * 1000 if exp != "kgram" else SEED_BASE + seed); np.random.seed(SEED_BASE + seed)
    if exp == "topology": model = B2Transformer(topology=cond)
    elif exp == "alpha": model = AlphaTransformer(alpha=float(cond))
    else: model = D3Transformer()
    model = model.to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    log(f"  {exp} {cond} seed {seed}: {n_params / 1e6:.1f}M params")
    mask = causal_mask(CFG["seq_len"], DEVICE)
    traj = {}
    def probe(step):
        p_nat, _, _ = survey_profile(model, mask, probe_nat); p_dist, _, _ = survey_profile(model, mask, probe_dist)
        base = eval_losses(model, eval_data)
        traj[step] = dict(profile_natural=p_nat.tolist(), profile_indist=p_dist.tolist(), stats_natural=two_arm(p_nat), stats_indist=two_arm(p_dist), eval_loss=float(np.mean(base)))
        s = traj[step]["stats_indist"]
        log(f"    step {step}: R_ex0 in-dist {s['R_ex0']:.3f} (natural {traj[step]['stats_natural']['R_ex0']:.3f}) E/M {s['early_over_mid']:.2f} L/M {s['late_over_mid']:.2f} eval {np.mean(base):.3f}")
    probe(0)
    last = train_model(model, train, seed, exp, probe)
    m = measure(model, seed, eval_data, probe_nat, probe_dist)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ck = ckpt_dir / f"{exp}_{cond}_s{seed}.pt"
    torch.save({k: v.detach().to("cpu", torch.float16) for k, v in model.state_dict().items()}, ck)
    s = m["stats_indist"]; r = m["regions_branch"]
    extra = {}
    if exp == "topology" and cond == "gated_middle": extra["final_gates"] = model.get_gate_values()
    if exp == "topology" and hasattr(model, "highway_proj"): extra["highway_weight_norm"] = float(model.highway_proj.weight.norm())
    if exp == "topology" and hasattr(model, "mh_projs"): extra["skip_weight_norms"] = {k: float(p.weight.norm()) for k, p in model.mh_projs.items()}
    if exp == "topology" and hasattr(model, "unet_projs"): extra["skip_weight_norms"] = {k: float(p.weight.norm()) for k, p in model.unet_projs.items()}
    log(f"    -> eval {m['baseline_eval_loss']:.3f}  R_ex0 in-dist {s['R_ex0']:.3f} (natural {m['stats_natural']['R_ex0']:.3f}) E/M {s['early_over_mid']:.2f} L/M {s['late_over_mid']:.2f} two-arm {s['two_arm_rule']} C {s['contrast_C']:.3f}  "
        f"branch waist {r['waist']['dL']:.3f} early {r['early']['dL']:.3f} late {r['late']['dL']:.3f} (w/e {m['waist_over_edge_branch']:.2f})  skip waist {m['skip_regions']['waist']['dL']:.3f}  ({time.time() - t0:.0f} s)")
    return dict(experiment=exp, condition=str(cond), seed=seed, params_M=round(n_params / 1e6, 1), final_train_loss=round(last, 4), trajectory=traj, **m, **extra,
                checkpoint=str(ck), estimator="survey_sigma1_v2 token-local J^T J, causal mask, 5 inputs x 8 positions", survey_version=sv.VERSION,
                intervention="branch rotation (attention and MLP outputs, linear dose interpolation as d3); identity skip; early exit", cfg=CFG,
                gpu=torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu", torch=torch.__version__, time_s=round(time.time() - t0),
                date=datetime.datetime.now().isoformat(timespec="seconds"))


# ================================================================ driver
def runs_for(exp):
    if exp == "topology": return [(t, s) for s in range(3) for t in TOPOLOGIES]
    if exp == "alpha": return [(a, s) for s in range(3) for a in ALPHAS]
    return [(k, s) for s in range(5) for k in K_VALUES]


def main(argv=None):
    ap = argparse.ArgumentParser(); ap.add_argument("--exp", default=os.environ.get("REPRO_EXP", "kgram"), choices=["topology", "alpha", "kgram"])
    args = ap.parse_args(argv); exp = args.exp
    OUT = ROOT / exp; OUT.mkdir(parents=True, exist_ok=True); CK = OUT / "ckpt"
    todo = runs_for(exp)
    log("=" * 70); log(f"reprofile v2: {exp}, {len(todo)} runs -> {OUT}; device {DEVICE}")
    if DEVICE == "cuda":
        p = torch.cuda.get_device_properties(0); log(f"GPU {p.name}, {p.total_memory / 1e9:.0f} GB")
    log("=" * 70)
    train, val = load_wikitext_corpus()
    from transformers import AutoTokenizer
    probe_nat = sv.natural_text_ids(AutoTokenizer.from_pretrained("gpt2"), 5)
    models = unigram_cum = None
    if exp == "kgram":
        all_ids = train.reshape(-1).tolist()
        counts = np.bincount(np.array(all_ids), minlength=VOCAB_SIZE).astype(np.float64); unigram_cum = np.cumsum(counts / counts.sum())
        models = build_ngram_models(all_ids)
    t0 = time.time()
    for i, (cond, seed) in enumerate(todo):
        path = OUT / f"{exp}_{cond}_s{seed}.json"
        if path.exists():
            try:
                d = json.load(open(path)); assert "profile_indist" in d
                log(f"[{i + 1}/{len(todo)}] {cond} s{seed}: done (R {d['stats_indist']['R_ex0']:.3f}), skipping"); continue
            except Exception:
                path.unlink()
        log(f"\n[{i + 1}/{len(todo)}] {exp} {cond} seed {seed}")
        if exp == "kgram":
            d = kgram_data(cond, seed, train, models, unigram_cum); tr, ev = d["train"], d["held"]; probe_dist = ev[-5:]
        else:
            tr, ev = train, val[:N_EVAL_BATCHES * EVAL_BATCH]; probe_dist = val[-5:]
        res = run_one(exp, cond, seed, tr, ev, probe_nat, probe_dist, CK)
        tmp = path.with_suffix(".tmp"); json.dump(res, open(tmp, "w"), indent=1); tmp.replace(path)
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    log(f"\nall done in {(time.time() - t0) / 3600:.2f} h"); summary(exp); export(exp)


def summary(exp):
    OUT = ROOT / exp
    rows = [json.load(open(f)) for f in sorted(OUT.glob(f"{exp}_*_s*.json"))]
    if not rows: return
    conds = TOPOLOGIES if exp == "topology" else ([str(a) for a in ALPHAS] if exp == "alpha" else [str(k) for k in K_VALUES])
    print("\n%-14s%3s %7s | %7s %7s %6s %6s %5s | %7s %7s %7s %5s | %7s %5s | %7s" % ("cond", "n", "eval", "R indist", "R nat", "E/M", "L/M", "two", "br w", "br e", "br l", "w/e", "skip w", "w/e", "C_fit"))
    for c in conds:
        rs = [r for r in rows if r["condition"] == c]
        if not rs: continue
        g = lambda f: np.mean([f(r) for r in rs])
        print("%-14s%3d %7.3f | %7.3f %7.3f %6.2f %6.2f %2d/%d | %7.3f %7.3f %7.3f %5.2f | %7.3f %5.2f | %7.3f" % (
            c, len(rs), g(lambda r: r["baseline_eval_loss"]), g(lambda r: r["stats_indist"]["R_ex0"]), g(lambda r: r["stats_natural"]["R_ex0"]),
            g(lambda r: r["stats_indist"]["early_over_mid"]), g(lambda r: r["stats_indist"]["late_over_mid"]), sum(r["stats_indist"]["two_arm_rule"] for r in rs), len(rs),
            g(lambda r: r["regions_branch"]["waist"]["dL"]), g(lambda r: r["regions_branch"]["early"]["dL"]), g(lambda r: r["regions_branch"]["late"]["dL"]), g(lambda r: r["waist_over_edge_branch"]),
            g(lambda r: r["skip_regions"]["waist"]["dL"]), g(lambda r: r["skip_waist_over_edge"]), g(lambda r: r["ufit_indist"]["C_fit"])))


def export(exp):
    OUT = ROOT / exp
    stage = ROOT / f"{exp}_json"; stage.mkdir(parents=True, exist_ok=True)
    for f in OUT.glob("*.json"): shutil.copy(f, stage / f.name)
    z = shutil.make_archive(str(ROOT / f"reprofile_{exp}_results"), "zip", str(stage))
    here = Path.cwd() / f"reprofile_{exp}_results.zip"
    if Path(z).resolve() != here.resolve(): shutil.copy(z, here)
    log(f"results zip: {z} and {here} -- on Drive already if ROOT is on Drive")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/markov_source.py
"""Random-transition m-th-order Markov sources over a small vocabulary, entropy matched.

A source is a transition table P[s, x] over the V^m contexts s = (x_{t-m}, ..., x_{t-1}) (state index
s = sum_j x_{t-m+j} V^(m-1-j), so the most recent token is the least significant digit). Rows are drawn
from Dirichlet(alpha) and tempered, P ∝ P0^(1/tau), with tau chosen by bisection so that the stationary
conditional entropy H(X_t | X_{t-m..t-1}) equals H_TARGET nats for every order and seed. Rows have full
support, so every chain is irreducible and aperiodic. Sequences start from the stationary distribution
over contexts, so no burn-in is needed and every position of a sequence has a full-order context in the
source (the model only sees the sequence, so its own floor at positions t < m is higher; the learned-gap
statistic uses positions t >= 8, valid for every order <= 8).
"""
import numpy as np

V_DEFAULT, H_TARGET, ALPHA = 8, 1.0, 0.5   # ALPHA is the Dirichlet parameter of the dense structure only


def stationary(P, V, m, tol=1e-12, max_iter=5000):
    S = P.shape[0]; base = V ** (m - 1)
    s = np.arange(S); nxt = np.stack([(s % base) * V + x for x in range(V)], axis=1)   # S x V next-state index
    pi = np.full(S, 1.0 / S)
    for it in range(max_iter):
        new = np.zeros(S)
        for x in range(V):
            new += np.bincount(nxt[:, x], weights=pi * P[:, x], minlength=S)
        new /= new.sum()
        if np.abs(new - pi).sum() < tol:
            pi = new; break
        pi = new
    return pi, it + 1


def cond_entropy(P, pi):
    return float(-(pi[:, None] * P * np.log(P)).sum())


def temper(P0, tau):
    L = np.log(P0) / tau; L -= L.max(axis=1, keepdims=True)
    P = np.exp(L); P /= P.sum(axis=1, keepdims=True)
    return P


def floors_by_order(P, pi, V, m):
    """H(X_t | last r tokens) for r = 0..m under the stationary law: the best loss a predictor that reads only
    the last r tokens can reach. r = 0 is the marginal entropy, r = m the conditional entropy of the source."""
    S = P.shape[0]; s = np.arange(S); out = []
    for r in range(m + 1):
        g = s % (V ** r)                                   # suffix-r index of every state
        joint = np.stack([np.bincount(g, weights=pi * P[:, x], minlength=V ** r) for x in range(V)], axis=1)   # P(suffix_r, x)
        marg = joint.sum(axis=1, keepdims=True); cond = joint / np.clip(marg, 1e-300, None)
        out.append(float(-(joint * np.log(np.clip(cond, 1e-300, None))).sum()))
    return out


def raw_table(V, m, seed, alpha, structure):
    """Untempered transition rows. dense: independent Dirichlet(alpha) rows, one per context (no structure
    below order m, so the gradient signal for a partial context of r < m tokens is of order V^-(m-r)/2 and
    the source is not learned at m >= 5 in the budget). hier: logits are a sum over r = 1..m of independent
    standard-normal tables indexed by the last r tokens, so every order contributes and the full order-m
    table is needed to reach the floor; the analogue of a back-off source with random statistics."""
    rng = np.random.default_rng(1_000_003 * m + 7919 * seed + 11)
    S = V ** m
    if structure == "dense":
        P0 = rng.dirichlet(np.full(V, alpha), size=S)
    else:
        s = np.arange(S); logits = np.zeros((S, V))
        for r in range(1, m + 1):
            G = rng.standard_normal((V ** r, V)); logits += G[s % (V ** r)]
        logits -= logits.max(axis=1, keepdims=True); P0 = np.exp(logits); P0 /= P0.sum(axis=1, keepdims=True)
    P0 = np.clip(P0, 1e-12, None); P0 /= P0.sum(axis=1, keepdims=True)
    return P0


def _match(P0, V, m, H_target):
    lo, hi = -4.0, 4.0     # log tau; H is increasing in tau (tau -> 0 deterministic rows, tau -> inf uniform rows)
    for _ in range(40):
        mid = 0.5 * (lo + hi); P = temper(P0, np.exp(mid)); pi, _ = stationary(P, V, m); H = cond_entropy(P, pi)
        if H < H_target: lo = mid
        else: hi = mid
    tau = np.exp(0.5 * (lo + hi)); P = temper(P0, tau); pi, n_it = stationary(P, V, m); H = cond_entropy(P, pi)
    marg = np.zeros(V)
    for x in range(V): marg[x] = (pi * P[:, x]).sum()
    return P, pi, tau, H, marg, float(-(marg * np.log(marg)).sum()), n_it


MARGINAL_SLACK = 0.10   # accept a source only if its marginal entropy is within this of ln V, so mutual information is matched too


def make_source(V=V_DEFAULT, m=1, seed=0, alpha=ALPHA, H_target=H_TARGET, structure="hier"):
    """Draw the table, temper it to H_target, and (low orders have few states, so the stationary marginal can be
    skewed) redraw with a sub-seed until the marginal entropy is within MARGINAL_SLACK of ln V."""
    S = V ** m
    for attempt in range(500):
        P0 = raw_table(V, m, seed + 100_000 * attempt, alpha, structure)
        pi0, _ = stationary(P0, V, m); H0 = cond_entropy(P0, pi0)
        P, pi, tau, H, marg, H_marg, n_it = _match(P0, V, m, H_target)
        if H_marg >= np.log(V) - MARGINAL_SLACK: break
    return dict(V=V, m=m, seed=seed, alpha=alpha, H_target=H_target, structure=structure, P=P.astype(np.float64), pi=pi, tau=float(tau), H_cond=H,
                H_cond_untempered=H0, H_marginal=H_marg, mutual_information=H_marg - H, marginal=marg, n_states=S, stationary_iters=int(n_it),
                draw_attempts=attempt + 1, floors_by_order=floors_by_order(P, pi, V, m))


def generate(src, n_seqs, seq_len, seed):
    """n_seqs sequences of seq_len tokens; initial context drawn from the stationary distribution."""
    P, pi, V, m = src["P"], src["pi"], src["V"], src["m"]; base = V ** (m - 1)
    rng = np.random.default_rng(seed)
    s = rng.choice(len(pi), size=n_seqs, p=pi)
    # the first m tokens are the digits of the initial context (most recent last)
    ctx = np.zeros((n_seqs, m), dtype=np.int64); ss = s.copy()
    for j in range(m - 1, -1, -1):
        ctx[:, j] = ss % V; ss //= V
    out = np.zeros((n_seqs, seq_len), dtype=np.int64); out[:, :m] = ctx[:, :seq_len]
    cum = np.cumsum(P, axis=1); cum[:, -1] = 1.0 + 1e-9
    for t in range(m, seq_len):
        u = rng.random(n_seqs)
        x = (u[:, None] > cum[s]).sum(axis=1)
        out[:, t] = x; s = (s % base) * V + x
    return out


def state_index(seqs, V, m, t):
    """context index for the token at position t (needs t >= m): digits seqs[:, t-m:t], most recent least significant."""
    s = np.zeros(seqs.shape[0], dtype=np.int64)
    for j in range(t - m, t): s = s * V + seqs[:, j]
    return s


def floor_per_position(src, seqs):
    """-ln P(x_t | x_{t-m..t-1}) under the true source for every position t >= m; nan before."""
    P, V, m = src["P"], src["V"], src["m"]; n, T = seqs.shape
    f = np.full((n, T), np.nan)
    for t in range(m, T):
        s = state_index(seqs, V, m, t); f[:, t] = -np.log(P[s, seqs[:, t]])
    return f


if __name__ == "__main__":
    import time, sys
    structure = sys.argv[1] if len(sys.argv) > 1 else "hier"
    for m in (1, 2, 3, 4, 5, 6):
        t0 = time.time(); src = make_source(m=m, seed=0, structure=structure)
        seqs = generate(src, 800, 128, seed=1); fl = floor_per_position(src, seqs)
        print(f"{structure} m={m}: states {src['n_states']:7d} tau {src['tau']:.3f} H0 {src['H_cond_untempered']:.3f} -> H {src['H_cond']:.5f}  "
              f"H_marg {src['H_marginal']:.3f} MI {src['mutual_information']:.3f} draws {src['draw_attempts']}  floors by order {np.round(src['floors_by_order'], 3).tolist()}  "
              f"held floor(t>=8) {np.nanmean(fl[:, 8:]):.4f}  ({time.time() - t0:.1f} s)")


In [ ]:
%%writefile /content/markov_v1.py
#!/usr/bin/env python3
"""
Second task family: random-transition m-th-order Markov sources, small vocabulary, entropy matched
===================================================================================================
The k-gram sweep varies context length, predictive dependence and difficulty together (the sources are
fitted to WikiText-103 over a 50,257-token vocabulary and the models never reach the entropy floor).
This family holds the vocabulary (V = 8) and the conditional entropy (H = 1.0 nats, matched by
temperature per source) fixed and varies only the order m, the number of preceding tokens the next token
depends on, m in {1, 2, 3, 4, 5, 6}, five seeds each (30 models). The transition logits of a context are
the sum over r = 1..m of independent standard-normal random tables indexed by the last r tokens (a
random back-off source): every order contributes, the full order-m table is needed to reach the floor,
and the floor a predictor reading only the last r tokens could reach, H(X_t | last r), is computed
exactly for every r (floors_by_order). A dense alternative, one independent Dirichlet row per context,
was tried first and rejected: it has no structure below order m, so the gradient signal for a partial
context of r tokens scales as V^-(m-r)/2, and a pilot at m = 5 sat at the uniform ceiling for 1,250
steps (markov_source.raw_table keeps both). The marginal entropy is held within 0.1 nats of ln V by
redrawing low-order tables, so mutual information is matched too (0.99-1.06 nats). Data are fresh every
batch (160,000 training sequences used once); the held-out set is 800 sequences.

Models, optimiser, schedule and every measurement are those of the k-gram sweep (reprofile_v2.py):
12-layer d = 512 pre-norm transformer (D3Transformer, tied 8-token embedding), AdamW wd 0.01, lr 3e-4,
500 warm-up, cosine, 10,000 steps, batch 16 x 128, dropout 0.1; canonical sigma_1 profile
(survey_sigma1_v2, in-distribution inputs; there is no natural-text probe for an 8-token model);
branch rotation by region at doses 0.25/0.5/1, fitted-waist rotation, identity skip by region,
per-block rotation and skip, early exit. Added for this family: the exact floor of the held-out set
under the true source and the learned gap (model loss minus floor over positions t >= 8, valid for every
m <= 8), a held-out learning curve every 500 steps, and the source statistics (tau, conditional and
marginal entropy, mutual information, number of contexts).

Outputs (MARKOV_OUT, on Drive): one JSON per run written atomically, resume-safe; fp16 checkpoint per run
in ckpt/; source and held-out cache in data/; results zip at the end (export()). Runs are ordered
seed-major (all orders at seed 0, then seed 1, ...) so that a partial sweep is a complete ladder.

  python markov_v1.py               full sweep (30 runs)
  python markov_v1.py --quick       learnability check: full model, orders 4/5/6, seed 0, 2,000 steps
  python markov_v1.py --plumbing    tiny models, 20 steps, end to end incl. summary and export

Needs survey_sigma1_v2.py, reprofile_v2.py and markov_source.py beside it.
"""
import os, sys, json, time, math, gc, datetime, argparse, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import survey_sigma1_v2 as sv
import reprofile_v2 as rp
import markov_source as ms

DEVICE = rp.DEVICE
ROOT = Path(os.environ.get("MARKOV_OUT", "markov_v1"))
V, H_TARGET, ALPHA = 8, 1.0, 0.5
ORDERS, N_SEEDS = [1, 2, 3, 4, 5, 6], 5
GAP_POS, EVAL_EVERY = 8, 500
N_HELD = rp.N_EVAL_BATCHES * rp.EVAL_BATCH      # 800 held-out sequences, as the k-gram sweep
rp.VOCAB_SIZE = V                                # every model, loss and probe in reprofile_v2 reads this global
CFG = rp.CFG
FULL = dict(CFG)                                 # the k-gram configuration, restored at every full-sweep start (the quick and plumbing modes mutate the shared dict)
log = rp.log


# ================================================================ data
def source_data(m, seed):
    """Source (tempered transition table, stationary distribution, entropies) and held-out set, cached on
    the persistent root; the training stream is regenerated deterministically at run start (20 MB per run
    otherwise)."""
    cache = ROOT / "data" / f"markov_m{m}_s{seed}.npz"
    if cache.exists():
        z = np.load(cache)
        src = dict(V=int(z["V"]), m=int(z["m"]), seed=int(z["seed"]), alpha=float(z["alpha"]), H_target=float(z["H_target"]), P=z["P"].astype(np.float64), pi=z["pi"],
                   tau=float(z["tau"]), H_cond=float(z["H_cond"]), H_cond_untempered=float(z["H_cond_untempered"]), H_marginal=float(z["H_marginal"]),
                   mutual_information=float(z["mutual_information"]), marginal=z["marginal"], n_states=int(z["n_states"]), stationary_iters=int(z["stationary_iters"]),
                   structure=str(z["structure"]) if "structure" in z else "hier", draw_attempts=int(z["draw_attempts"]) if "draw_attempts" in z else 1,
                   floors_by_order=z["floors_by_order"].tolist() if "floors_by_order" in z else None)
        return src, z["held"].astype(np.int64), z["floor_held"]
    t0 = time.time()
    src = ms.make_source(V=V, m=m, seed=seed, alpha=ALPHA, H_target=H_TARGET)
    held = ms.generate(src, N_HELD, CFG["seq_len"], seed=10_000 * m + seed + 777)
    floor_held = ms.floor_per_position(src, held).astype(np.float32)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache, held=held.astype(np.uint8), floor_held=floor_held, P=src["P"].astype(np.float32), **{k: v for k, v in src.items() if k != "P"})
    log(f"  source m={m} seed {seed} ({src['structure']}): {src['n_states']} contexts, tau {src['tau']:.3f}, H {src['H_cond']:.4f} (untempered {src['H_cond_untempered']:.3f}), "
        f"H_marg {src['H_marginal']:.3f}, MI {src['mutual_information']:.3f}, floors by order {np.round(src['floors_by_order'], 3).tolist()}, "
        f"held floor(t>={GAP_POS}) {np.nanmean(floor_held[:, GAP_POS:]):.4f} ({time.time() - t0:.0f} s)")
    return src, held, floor_held


def training_stream(src, seed, n_steps):
    return torch.from_numpy(ms.generate(src, n_steps * CFG["batch_size"], CFG["seq_len"], seed=10_000 * src["m"] + seed + 1))


# ================================================================ measurement
@torch.no_grad()
def tail_loss(model, held):
    """Held-out loss over targets t >= GAP_POS, the positions at which every order <= GAP_POS has its full context."""
    model.eval(); model.drop.p = 0.0; mask = rp.causal_mask(CFG["seq_len"], DEVICE); tot = 0.0; n = 0
    try:
        for b in range(0, len(held), rp.EVAL_BATCH):
            ids = held[b:b + rp.EVAL_BATCH].to(DEVICE); lg = model(ids, attn_mask=mask)
            tot += F.cross_entropy(lg[:, GAP_POS - 1:-1].reshape(-1, V), ids[:, GAP_POS:].reshape(-1)).item(); n += 1
        return tot / max(n, 1)
    finally:
        model.drop.p = 0.1; model.train()


def measure(model, seed, held, probe_dist):
    """rp.measure without the natural-text profile (an 8-token model has none)."""
    L = len(model.blocks); out = {}
    mask = rp.causal_mask(CFG["seq_len"], DEVICE)
    prof, samp, it1 = rp.survey_profile(model, mask, probe_dist)
    out["profile_indist"] = prof.tolist(); out["samples_indist"] = [[round(x, 4) for x in s] for s in samp]
    out["stats_indist"] = rp.two_arm(prof); out["ufit_indist"] = rp.fit_ushape(prof); out["power_iters"] = [it1]
    base = rp.eval_losses(model, held); out["baseline_eval_loss"] = float(np.mean(base))
    Qa = {li: rp.haar(CFG["d_model"], rp.SEED_BASE + li * 100 + seed) for li in range(L)}
    Qf = {li: rp.haar(CFG["d_model"], rp.SEED_BASE + li * 100 + 10 + seed) for li in range(L)}
    regions = {}
    for rn, blk in (("early", rp.EARLY), ("waist", rp.WAIST), ("late", rp.LATE)):
        for dose in rp.DOSES:
            regions[f"{rn}@{dose}"] = rp.dstat(rp.eval_losses(model, held, rp.rotate_branches(model, blk, Qa, Qf, dose)), base)
        regions[rn] = regions[f"{rn}@1.0"]
    out["regions_branch"] = regions
    edge = max(regions["early"]["dL"], regions["late"]["dL"], 1e-9); out["waist_over_edge_branch"] = regions["waist"]["dL"] / edge
    fit = out["ufit_indist"]; fr = {}
    for rn in ("waist", "early", "late"):
        if fit[rn]:
            fr[rn] = rp.dstat(rp.eval_losses(model, held, rp.rotate_branches(model, fit[rn], Qa, Qf, 1.0)), base)
    out["regions_branch_fitted_waist"] = fr
    out["skip_regions"] = {rn: rp.dstat(rp.eval_losses(model, held, rp.skip_blocks(model, blk)), base) for rn, blk in (("early", rp.EARLY), ("waist", rp.WAIST), ("late", rp.LATE))}
    sedge = max(out["skip_regions"]["early"]["dL"], out["skip_regions"]["late"]["dL"], 1e-9); out["skip_waist_over_edge"] = out["skip_regions"]["waist"]["dL"] / sedge
    out["per_block_branch_dL"] = [rp.dstat(rp.eval_losses(model, held, rp.rotate_branches(model, [li], Qa, Qf, 1.0)), base)["dL"] for li in range(L)]
    out["per_block_skip_dL"] = [rp.dstat(rp.eval_losses(model, held, rp.skip_blocks(model, [li])), base)["dL"] for li in range(L)]
    out["early_exit_loss"] = rp.early_exit(model, held)
    return out


# ================================================================ training
def train_model(model, stream, seed, held, probe_cb, curve_cb):
    """The k-gram optimiser (AdamW wd 0.01, cosine with 500 warm-up, clip 1.0) on fresh batches."""
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    mask = rp.causal_mask(CFG["seq_len"], DEVICE); bs = CFG["batch_size"]; model.train()
    t0, last = time.time(), None
    for step in range(CFG["n_steps"]):
        if step in rp.PROFILE_AT: probe_cb(step)
        if step % EVAL_EVERY == 0: curve_cb(step)
        ids = stream[step * bs:(step + 1) * bs].to(DEVICE)
        for pg in opt.param_groups: pg["lr"] = rp.get_lr(step, True)
        loss = F.cross_entropy(model(ids, attn_mask=mask)[:, :-1].reshape(-1, V), ids[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        last = loss.item()
        if (step + 1) % 2500 == 0: log(f"    step {step + 1}/{CFG['n_steps']} loss {last:.3f} ({time.time() - t0:.0f} s)")
    curve_cb(CFG["n_steps"])
    return last


def run_one(m, seed, ckpt_dir, quick=False):
    t0 = time.time()
    src, held_np, floor_held = source_data(m, seed)
    held = torch.from_numpy(held_np); probe_dist = held[-5:]
    floor_tail = float(np.nanmean(floor_held[:, GAP_POS:])); floor_all = float(np.nanmean(floor_held[:, 1:]))
    torch.manual_seed(rp.SEED_BASE + seed); np.random.seed(rp.SEED_BASE + seed)
    model = rp.D3Transformer().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    log(f"  markov m={m} seed {seed}: {n_params / 1e6:.1f}M params; floor(t>={GAP_POS}) {floor_tail:.4f}")
    stream = training_stream(src, seed, CFG["n_steps"])
    mask = rp.causal_mask(CFG["seq_len"], DEVICE)
    traj, curve = {}, []
    def probe(step):
        p, _, _ = rp.survey_profile(model, mask, probe_dist); base = rp.eval_losses(model, held)
        traj[step] = dict(profile_indist=p.tolist(), stats_indist=rp.two_arm(p), eval_loss=float(np.mean(base)), tail_loss=tail_loss(model, held))
        s = traj[step]["stats_indist"]
        log(f"    step {step}: R_ex0 {s['R_ex0']:.3f} E/M {s['early_over_mid']:.2f} L/M {s['late_over_mid']:.2f} eval {traj[step]['eval_loss']:.3f} gap {traj[step]['tail_loss'] - floor_tail:+.4f}")
    def on_curve(step):
        tl = tail_loss(model, held); curve.append([step, round(tl, 5)])
        if step % 2000 == 0: log(f"    step {step}: held(t>={GAP_POS}) {tl:.4f} gap {tl - floor_tail:+.4f}")
    if not quick: probe(0)
    last = train_model(model, stream, seed, held, probe if not quick else (lambda s: None), on_curve)
    tl = tail_loss(model, held); gap = tl - floor_tail
    src_stats = dict(V=V, order=m, structure=src["structure"], n_states=src["n_states"], tau=src["tau"], H_cond=src["H_cond"], H_cond_untempered=src["H_cond_untempered"],
                     H_marginal=src["H_marginal"], mutual_information=src["mutual_information"], floors_by_order=src["floors_by_order"], draw_attempts=src["draw_attempts"],
                     floor_tail=floor_tail, floor_all_positions=floor_all, uniform_ceiling=float(np.log(V)), stationary_iters=src["stationary_iters"])
    below_prev = (src["floors_by_order"][-2] - tl) if src["floors_by_order"] and m >= 1 else float("nan")   # margin below the best order-(m-1) predictor
    if quick:
        log(f"    -> quick m={m}: held(t>={GAP_POS}) {tl:.4f} floor {floor_tail:.4f} gap {gap:+.4f}; below the order-{m - 1} floor by {below_prev:+.4f} ({time.time() - t0:.0f} s)")
        return dict(experiment="markov_quick", condition=str(m), seed=seed, tail_loss=tl, learned_gap=gap, margin_below_prev_order=below_prev, learning_curve=curve, source=src_stats, cfg=dict(CFG),
                    gpu=torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu", torch=torch.__version__, time_s=round(time.time() - t0), date=datetime.datetime.now().isoformat(timespec="seconds"))
    mres = measure(model, seed, held, probe_dist)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ck = ckpt_dir / f"markov_{m}_s{seed}.pt"
    torch.save({k: v.detach().to("cpu", torch.float16) for k, v in model.state_dict().items()}, ck)
    s = mres["stats_indist"]; r = mres["regions_branch"]
    log(f"    -> eval {mres['baseline_eval_loss']:.3f} tail {tl:.4f} gap {gap:+.4f} (below order-{m - 1} floor by {below_prev:+.4f})  R_ex0 {s['R_ex0']:.3f} E/M {s['early_over_mid']:.2f} L/M {s['late_over_mid']:.2f} two-arm {s['two_arm_rule']} C {s['contrast_C']:.3f}  "
        f"branch waist {r['waist']['dL']:.3f} early {r['early']['dL']:.3f} late {r['late']['dL']:.3f} (w/e {mres['waist_over_edge_branch']:.2f})  skip waist {mres['skip_regions']['waist']['dL']:.3f}  ({time.time() - t0:.0f} s)")
    return dict(experiment="markov", condition=str(m), seed=seed, params_M=round(n_params / 1e6, 1), final_train_loss=round(last, 4), trajectory=traj,
                tail_loss=tl, learned_gap=gap, margin_below_prev_order=below_prev, learning_curve=curve, source=src_stats, **mres,
                checkpoint=str(ck), estimator="survey_sigma1_v2 token-local J^T J, causal mask, 5 inputs x 8 positions", survey_version=sv.VERSION,
                intervention="branch rotation (attention and MLP outputs, linear dose interpolation as d3); identity skip; early exit",
                data=f"random-transition order-{m} Markov source ({src['structure']}), V={V}, tempered to H={H_TARGET} nats; fresh batches; {N_HELD} held-out sequences",
                cfg=dict(CFG), gpu=torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu", torch=torch.__version__, time_s=round(time.time() - t0),
                date=datetime.datetime.now().isoformat(timespec="seconds"))


# ================================================================ driver
def runs():
    return [(m, s) for s in range(N_SEEDS) for m in ORDERS]


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true"); ap.add_argument("--plumbing", action="store_true")
    args = ap.parse_args(argv)
    global ORDERS, N_SEEDS
    if args.plumbing:
        CFG.update(n_layers=6, d_model=32, n_heads=4, d_ff=64, n_steps=20, warmup_steps=5); rp.PROFILE_AT[:] = [10]
        rp.EARLY, rp.WAIST, rp.LATE = [0, 1], [2, 3], [4, 5]; rp.N_EVAL_BATCHES = 5
        ORDERS, N_SEEDS = [1, 3], 1
    if args.quick:
        CFG.update(n_steps=2000); ORDERS, N_SEEDS = [int(x) for x in os.environ.get("MARKOV_QUICK_ORDERS", "4,5,6").split(",")], 1
    if not (args.quick or args.plumbing):
        CFG.update(FULL); rp.EARLY, rp.WAIST, rp.LATE = [0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]; rp.N_EVAL_BATCHES = 50; rp.PROFILE_AT[:] = [5000]
    exp = "markov_quick" if args.quick else "markov"
    OUT = ROOT / exp; OUT.mkdir(parents=True, exist_ok=True); CK = OUT / "ckpt"
    todo = runs()
    log("=" * 70); log(f"markov v1: {exp}, {len(todo)} runs -> {OUT}; device {DEVICE}; V={V}, H={H_TARGET}, orders {ORDERS}, seeds {N_SEEDS}, n_steps {CFG['n_steps']}")
    if DEVICE == "cuda":
        p = torch.cuda.get_device_properties(0); log(f"GPU {p.name}, {p.total_memory / 1e9:.0f} GB")
    log("=" * 70)
    t0 = time.time()
    for i, (m, seed) in enumerate(todo):
        path = OUT / f"{exp}_{m}_s{seed}.json"
        if path.exists():
            try:
                d = json.load(open(path)); assert ("profile_indist" in d) or args.quick
                assert d["cfg"]["n_steps"] == CFG["n_steps"], f"{path.name} was trained for {d['cfg']['n_steps']} steps, this sweep uses {CFG['n_steps']}: re-running it"
                log(f"[{i + 1}/{len(todo)}] m={m} s{seed}: done (gap {d['learned_gap']:+.4f}, {d['cfg']['n_steps']} steps), skipping"); continue
            except Exception as e:
                log(f"  {e}"); path.rename(path.with_suffix(".old.json"))
        log(f"\n[{i + 1}/{len(todo)}] markov m={m} seed {seed} ({CFG['n_steps']} steps)")
        res = run_one(m, seed, CK, quick=args.quick)
        tmp = path.with_suffix(".tmp"); json.dump(res, open(tmp, "w"), indent=1); tmp.replace(path)
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    log(f"\nall done in {(time.time() - t0) / 3600:.2f} h"); summary(exp); export(exp)


def summary(exp="markov"):
    OUT = ROOT / exp
    rows = [json.load(open(f)) for f in sorted(OUT.glob(f"{exp}_*_s*.json"))]
    if not rows: print("no results yet"); return
    if exp == "markov_quick":
        print("\n%-6s%3s %8s %8s %8s | learning curve (step: held loss, t>=8)" % ("order", "n", "floor", "held", "gap"))
        for r in sorted(rows, key=lambda r: (int(r["condition"]), r["seed"])):
            print("%-6s%3d %8.4f %8.4f %+8.4f | %s" % (r["condition"], 1, r["source"]["floor_tail"], r["tail_loss"], r["learned_gap"],
                  "  ".join(f"{s}:{v:.3f}" for s, v in r["learning_curve"][::2])))
        return
    print("\n%-6s%3s %7s %7s %7s %7s | %7s %6s %6s %5s | %7s %7s %7s %5s | %7s %5s | %7s" % ("order", "n", "floor", "held", "gap", "m-1 mrg", "R_ex0", "E/M", "L/M", "two", "br w", "br e", "br l", "w/e", "skip w", "w/e", "C_fit"))
    for m in sorted(set(int(r["condition"]) for r in rows)):
        rs = [r for r in rows if int(r["condition"]) == m]
        g = lambda f: np.mean([f(r) for r in rs])
        print("%-6d%3d %7.4f %7.4f %+7.4f %+7.4f | %7.3f %6.2f %6.2f %2d/%d | %7.3f %7.3f %7.3f %5.2f | %7.3f %5.2f | %7.3f" % (
            m, len(rs), g(lambda r: r["source"]["floor_tail"]), g(lambda r: r["tail_loss"]), g(lambda r: r["learned_gap"]), g(lambda r: r.get("margin_below_prev_order", float("nan"))),
            g(lambda r: r["stats_indist"]["R_ex0"]), g(lambda r: r["stats_indist"]["early_over_mid"]), g(lambda r: r["stats_indist"]["late_over_mid"]),
            sum(r["stats_indist"]["two_arm_rule"] for r in rs), len(rs),
            g(lambda r: r["regions_branch"]["waist"]["dL"]), g(lambda r: r["regions_branch"]["early"]["dL"]), g(lambda r: r["regions_branch"]["late"]["dL"]), g(lambda r: r["waist_over_edge_branch"]),
            g(lambda r: r["skip_regions"]["waist"]["dL"]), g(lambda r: r["skip_waist_over_edge"]), g(lambda r: r["ufit_indist"]["C_fit"])))
    print("n_steps per run:", sorted(set(r["cfg"]["n_steps"] for r in rows)), "| output folder:", OUT)
    print("per-run gaps:", "  ".join(f"m{r['condition']}s{r['seed']}:{r['learned_gap']:+.3f}" for r in sorted(rows, key=lambda r: (int(r["condition"]), r["seed"]))))


def export(exp="markov"):
    OUT = ROOT / exp
    stage = ROOT / f"{exp}_json"; stage.mkdir(parents=True, exist_ok=True)
    for f in OUT.glob("*.json"): shutil.copy(f, stage / f.name)
    z = shutil.make_archive(str(ROOT / f"{exp}_v1_results"), "zip", str(stage))
    here = Path.cwd() / f"{exp}_v1_results.zip"
    if Path(z).resolve() != here.resolve(): shutil.copy(z, here)
    log(f"results zip: {z} ({os.path.getsize(z) / 1e6:.2f} MB) and {here} -- on Drive already if MARKOV_OUT is on Drive")


if __name__ == "__main__":
    main()


In [ ]:
# Optional learnability check: full model, orders 4/5/6, seed 0, 2,000 steps each (~8 min on an A100). Writes to markov_v1/markov_quick/.
import runpy, sys, os
os.environ["MARKOV_QUICK_ORDERS"] = "4,5,6"
sys.argv = ["markov_v1.py", "--quick"]
runpy.run_path("/content/markov_v1.py", run_name="__main__")

In [ ]:
# The sweep (resume-safe; finished 10,000-step runs are skipped, 2,000-step files are renamed *.old.json and re-run).
import sys, os, importlib
sys.argv = ["markov_v1.py"]
for k in [k for k in sys.modules if k in ("markov_v1", "reprofile_v2", "markov_source", "survey_sigma1_v2")]: del sys.modules[k]   # fresh modules, no state from the learnability cell
import markov_v1 as mv
mv.ORDERS = [int(x) for x in MARKOV_ORDERS.split(",")]
mv.CFG.update(mv.FULL); assert mv.CFG["n_steps"] == 10000, mv.CFG
print("output folder:", mv.ROOT, "| n_steps:", mv.CFG["n_steps"])
mv.main([])

In [ ]:
# Summary table from the JSONs on disk (correct after a resume); prints the folder and the n_steps found
import sys, importlib
sys.path.insert(0, "/content")
import markov_v1 as mv; importlib.reload(mv)
mv.summary("markov")

In [ ]:
# Export again by hand if needed (the run cell already exports at the end)
mv.export("markov")